# Lab 6 – Errors, debugging, generators, decorators (src/cleaning.py)
**Lecture 6:** iterators · generators · decorators · try/except · debugging · the Python debugger
**Project module today:** `src/cleaning.py` (+ your own `to_number` and `to_bool` in `src/loader.py`)

Real data is never clean: empty cells, `"abc"` instead of a number, a negative age. Today your project learns to **survive bad data** and to **say what it removed and why**.

| Part | What you do | Data |
|---|---|---|
| 1 | Setup – checks Lab 2, Lab 4 and Lab 5 | your project |
| 2 | Lecture 6 step by step | 🧪 TEST DATA (broken Pokémon rows) |
| 3 | Your project: `loader.py`, `cleaning.py`, `main.ipynb` | **your data** |
| 4 | Result check → `docs/lab06_check.md` | your project |

**How to work**
- Run the cells from top to bottom (Shift+Enter).
- `...` = a place for **your** code. Above every `...` there is a comment that says in words **what to write in this line**.
- At the end of each code cell: **💡 Stuck?** – it tells you where you already did something similar.
- After each task: a **✔️ Check** cell.
- This notebook **does not change anything** in your project. You change your files yourself.

**Working on your computer (Jupyter / VS Code) without internet?** No problem – the TEST DATA is built into the Setup cell. Put this notebook **inside your project folder** (next to `src/` and `main.ipynb`). If Setup still does not find your project, write its path in `PROJECT_PATH` at the top of the Setup cell, with `/` (e.g. `"C:/Users/Ola/Desktop/PyDS_project"`).

🏆 **For those who finish today:** show the teacher **✅ 6/6** at the end.

## 1. Setup – run this first
It finds your project and checks Lab 2, Lab 4 and Lab 5. It **only reads**.
- **✅** → do all parts.
- **❌ numbered list** → each point says **which lab and step** fixes it. Part 2 works anyway (TEST DATA); parts 3–4 wait until the list is empty.

In [4]:
# @title ⚙️ Setup Lab 6 – run this cell first (click ▶ on the left)
# This cell only READS: it connects to your project and checks Lab 2 (loader.py),
# Lab 4 (stats.py) and Lab 5 (models.py + main.ipynb). It creates or changes NOTHING
# in your project. It also prepares the 🧪 TEST DATA (Pokémon) – built in, no internet needed.
import sys, os, json, csv, io, re, importlib, contextlib, datetime, types, time, traceback
from pathlib import Path
sys.dont_write_bytecode = True          # do not leave __pycache__ files in your project

PROJECT_NAME = "PyDS_project"

# The TEST DATA is built into this cell – it works without internet.
# (Optional) to use your own copy of pokemon_data.csv, write its path here:
TEST_FILE = ""      # @param {type:"string"}
#   use / (not \) in the path, e.g.  "E:/pokemon_data.csv"   "C:/Users/Ola/Desktop/pokemon_data.csv"   "/Volumes/USB/pokemon_data.csv"
# ▶ Project has another name or place? Write its folder name (e.g. PyDS_basketball_final) or the full path:
PROJECT_PATH = ""   # @param {type:"string"}
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
    PROJECT = Path("/content/drive/MyDrive") / PROJECT_NAME
except ImportError:                     # VS Code / PyCharm / Jupyter on your computer
    IN_COLAB = False
    PROJECT = Path.cwd() / PROJECT_NAME
    for _folder in [Path.cwd(), *Path.cwd().parents]:
        if (_folder / "src" / "loader.py").exists():
            PROJECT = _folder
            break
if PROJECT_PATH.strip():
    PROJECT = Path(PROJECT_PATH.strip().strip('"')).expanduser()
    if not PROJECT.is_absolute():       # only a folder name, e.g. "PyDS_basketball_final"
        PROJECT = (Path("/content/drive/MyDrive") if IN_COLAB else Path.cwd()) / PROJECT

# ---------------------------------------------------------------- 🧪 TEST DATA
TEST_URL = "https://raw.githubusercontent.com/KeithGalli/pandas/master/pokemon_data.csv"

# Built-in copy of the TEST DATA (pokemon_data.csv, compressed) – no internet needed. Do not change this line.
_BUILT_IN_TEST = "H4sIAMmz/GoC/5V925LcxrXl+/kKRkxMzExEHQduicsjZYqSbdGiRVmy9YbuQldjGl0oo6rYLP7R+Y75sVn7kpk7UdUaT7iJpui9Eom87Nz3/G+bv/bPw+bny2F4k8uvYvP9x83b06m/f9q8Gx6G/XHYfDr84c3b0xP/xr/h9zBsN98N+2HpT+O83/ww7Ib9tl8u/5FvvjlPd/2xPy+b75b+eNx8nMcjSCq3qTr6qR394D/zzfu3P3z69j+KzZ8+X64Bdbapi01dbtqMfvCfHlBufhn252sEERabttzkWcZ/2luYD8Ouf/NqAwwt0ERR4E/SRLX542O/PPf77bBs3o/LsNmU3cYVm6qk3rmMvssTOyEepgHtCrFrN3VFz5YpTcs1E49f+2UrtO+ny7jfbRrQVvTMs27TOuncNYa/KPzXm39IG++Wfod3E7oELM/597/dyj/XPckz6Yrr0FjaSrP59K/zuJymYfNrf8LobKpqU7X0lTQqFY2Qp21BspxmS4wWZZplVDBCnrjbfDP1xxNmJxA3nZ9h/hTHfbom56+5Bc6zkic2Lx1/h4Hn2eaPRHcYB6zi3YYWKYYMhKDHj1mzGM0Pw6k/zFuhdEzgQMk/ZRybvADFCa0+LIM0q0Nac4MAdrzKGgMpN79ig01C7vcPd6TMtC/OkFebv/RP532fkHM/qFvSIdN1t/kGrS/jNFkAxh4dqbhT1J2bABnV2+jcMTyXlW3Hqt58HLe74bL564ztM/kBkHfpd6GvdUQ0gphPp3kFqmWv8afhp8kjqPWgFaQt9YMaGeXsFoa/7PfwrSwZ+lWYBrrNTz2Y5an3IJoifErph70pIqPLiHi8xxoL1PiSNlf2gc51TaTOic/2y/xyPW6gp4WZ049ZOOBZ728hZG6J8eb0Y7cuWN23T/0+sEDqMq3Mil7jKupfIK02b5e7+SmQ1rwHa2bptC1Nqw7j+tTfP543307D/WkZ77llx/MtU9cZ8hoDM6bUNVM4p9sjzw05+A248PFxGV7AwOfzfss7EFPc8l4tZSV68lbIwQweA3mjHCyXXegs7y66zV/H7bz0+zf/8/2AcRz+V/hm+oKGmb5slrgU8FJGjdiInrqRM6zhBZsscUwcUf/rPAx7T659wwd3OMYa/Z7GgIrYsQ9Jt6p64xrfp4Q7lKXv1hwnLqdl6Xy3zKeXFVM/0apJe9XSuik2TcN94p4FEA67aXjox+Wyec9P+nDa2y2f4gn7KWshvgN7U+JOtmZJf+nSk77Z/HKeDuMXf962NOKVP2xtx1t0fA+WPA1HJUaDGLvGSdezZNnj4P7zuNtNl8P54SHsFu4OHQqVU35f8F+CBJBtfmXU6RpVZfrRLfMmw//Q49/Od/3JD2jC/oSnV5ndaFWx+W6erhGN04Oi5hkwGwjn64/b7cjrO5W5HLfc8FiZUwn7+7tpnp+vRS6eC5lis4fQ0i/jNBymM8TFBCOdotWQy441IDD+HrR8UAio5Oad001XmI9uhBj739ATF3Aq/rVZ8gUtSXTzvj8l55A/Gyr/lkDfMf3zfHq0ABlNYTbpmKKRd+NuGk6nwDRyf8SX3HRnxL188+68A+eaA7EcFs5z9tysI7CPD8P8gp74YyAsBtm/thvl5uOwHMd+H4hljuqwBXLTj2rz8XjZniG8e+EqY27lhTGzzPBXLDNLKwI0JKIuTKqnrTcf+v0TzvD34+7xRKuR+tn6w7ty9hwCZ/m4jM9DfxgMfS3yWs1db7Jk+FoatpdpPD0OXlp2up8a2e2m9Q7n0H2/x3ZX2k4YeRukftM0gB/naXzpd0E2zXSBXI01+CLTPo7L5KlljGt3fWqBszM1dKBHpQ4f23EfhLF1iXAHAebt3dLTNN0/0kGnLEYGxyVLsCbJbtsn5EEOpAXF4oz5Vrd5O/VP/df+OQJc4EelLFmzEA2ABaDX0DR1jaDN/q5pUdw/zgczyU2mq17WhWH8OAWZ+smuCZ2yMMdmw9Yt0/fPtnkanDLzuoKzyxlyyDfDNB0P2H+nlEeJbCC8J5UN8A8QtQ/j/g7QFTN0ys5az60CKAczhJgyDEDd0CBlV8pnmZnHkfvzsIdiPc9+dUXZXqbVheEIoFJBy3lYocJZ0Cp3sRoZ+Pswb8/bYfPTjA2uTKkKm6TMVJcICEdf8hkK65JAnEy7HIpVMqM4XcFAhueE3nemzFS0MFMK4eHjvL9AWNY9nukcyg4zKx886Kf+MG5JZBPaOKQqjJt1D+Hz0zS/HGht6RDp+u08k6y8chL00Iwxd8u8hvDZx/Jmtj50Ioi3y+820IalahuA3tjv9sPzCB0gCLufTgMmt/BLtHOrswtMmVEnzPoKJNtGt3W63qC3vO+Xh+F0//g/tiuNAEdCLSJpyxKqQVWbd1g4a72rdLoPZFMb9RD/DACdfCulQ5iynlTp8mzrzadhsDy28t/cJEwAUvC74QV2DM+9/3Q/KENt/SFiP5nOERw8YY/oYnFeKjYCR9ttPpyjMsPrKfOiEnXWcDq88dMjOARZfrTPZebpREw0TKXLIeLOl+MpUFOvnUxT6xmK6TbE/e9gqZgum+8e52MQ+8rA6rNMxYmAKDff9+c9tZ1AqsDsc+e3rocQS9jv+hWi9gIQLVa/8NcQXu2vo6ljeqCbtY7//HEP0d1yBxEVctFfV4NQw2Q1v3yFlSScPnQitMx4SrZORF0aivL3lwPpNJ6WZNZSFQn6+jqq0l27+cvS391dzOTRXHfePGKnutv8Bct3ijPt9LPyYHQwtg3MzC/zBHvWndFdK/48kbZcaioj64MQztshVXed5W6VhRSbb78Mu939+RSEb/3qOlOeXntjSkSVgjqjcyuUiCYk3hVO+UCEwcx5vpv3g9WsZe1W/mCPxA6MCZPWhwlWQZ1WkeO/WUNQVm++H0/P834arBDg5MiApFvK+msbg2kUA1FgvwJlbHdQkNGT86zd/DDeP42nM0i92NwFVUjEATtWmPT54cFovWxl4WGqs9WxRy+D0PDVUuvpVYjqlIirZHn96fHyOC9Bl+YdIYygc/4oti8oCLGdUwBzKBX85Cy20w2jIYy3sNYHMxuaxBdvpO9Zsshz2Ax/7ve7Yep1afhloJbdavUJDoIodv/x6dHoIbyFePfLGuxeQzD7eKUBWoL0Rn9C2L0Cy+H383IcerNvK3/MFS7tIoxCQ79derNtZQI7p2p5pG1Z8yHTi9cJmF/UXgirS0PcUcNsEYlqUleoXEFbz1iQMTWfTvCDnG2PnSozLu0FGfhA+zyuhZbam7K8lTu3KMgCyx/efMApFwBighAdUg8l6omdD9j5Pt1foF4t1gbdBMXJubVQlcPe9+fL/gsfXqFrzrOA3I9tpHfCz/q789evlrU5MqFCJ1Sek7yDVIjdc79EKY+kn8Z/emcmoiCr8P44yhfwh+racd5ocU2rdt0AixbR3InQpFs36VWL3XFe5mNYqWouFCbYpObIHPZCfATsncvBTzuL47yhWLtrrXE/23x3gaVji+ZVa5Q+dfJB4Gi6F9r8JkoO47SJd/3yJOqZY2dRo8KnbSHf/NAfyBwTZRL1CAVWZDcUjI3vRmOA55M4/Ym0OGuGz0Ni1g565+qAgYnxFzhOFnKM6WjlZRBCch7k2tK7zZ9xwg5W9vVaeQBYuSOHifH91C/R81YLgQrKeZa2T0rJciFfmTGwiIniyg6Ww874I1yAl5MqVvIFYrjhFcKzbk8LmBqBOGKnW4SeEYU6oRLDDB3+f8Euwtjbd3ge69b6aA4Lo9AfjhZRZ8rPnTfrWadRsXk7QAjp70+XSVDe3xF0WGFEydgmMLEZvN6KKA0qGSYeK3Cj/bxM/Zd4FuhkeptWojblFfke4Do5Q+Kjheu3THBo6uZhHvDzT38niNv81h9oi4RlE1EKLHTvZxEFhoTVtngrcoTo7OpbOgNpyNcKqWT0PlcyUpPX0wuMyae3SgWeHehheILeVXsXlfUFwm4pVKQxKr3vFLl0K/70a4c3jfeH4eX0YkRkEsDUTJvDUqjGFP8dAcDTKn+FMzkcMl704lb8sKEZ/v3/auefq16oURQaBlsLKgMn1+qLJc9u/wkfWkDyGZ/g4jgFaUaiDerwBBcO5OXmm/4CCfQhiD7spWkz74XODHG1oa/Yj2dvKlcjKY926Qc90sMVckEkBNzh06sxApG63vwNhP3n/tUQgUjbUHzG4zQfx8DXbsUHRAAOsfk0b0fjaJftVal7Hn2K1N3mj8t8j2X5YrRy1Z7ZP4/2AjU+5T1iP6Zx15+CnuTPdrLSsJO+sQgIO7BjLcMpOkodua2wgEvvcInU+Nt5scTiCNNNJZpbIIYuPMOsP195bmtv58KJgLF2FgRH13x/gs15hVHN37FCAZ20sSCHSJft5a63okSVRZejCFKRviZ6Mt0bejke1BSfMx+KAAixMEbiaH9a+f5rY7EubZdaMMaRJYHUq9EZ00tlATzR186l1sVABDVdBQz+CTsMGtjs5drAUkmRaunH8RDjAyKKBI796bzs1yDioVhO+CEH3QoFq/rKFVx4253Ynuz+hK5P3sSH3vsSNeKh9UekGVzYRP9Ezru7K+ddJ4eN07lMQA67aAcLsX+BRIOIRbFOVy2+hGhPQSKPM155OYt2iJ0OmET/2p/OkcuGReWyaA6zixC76h83EOITDBaxzn4DyaewVh/MmDpvEq29VS+Qo5cQn/qHh4uhb7wxI9gaIn2+eft8QOCQPWrVn6baS5fuC4MQEeIKrgeduFGYq6hQkLy4YIv/fDwGR6a6I7twrts9D5aEcaDIFZW7eZIaH09TeDthBEDq+Hp+vsZoy8EvmbzFbT6dt/PLPG9VfhM3jQhjIunazQVbKLmRTjNC6pSTytg14SPs9MM28v18OIwH/eJony19MI1I3hGB+Ien8WDcvWZdGiHdnjYwjv75/Ax378NDimmMMqvCdACht2/HA95itAD19nrlN9LiPDjDochGEZ65Mlv/ROKCiB8mRD6YeW7UuMYNJ+Tl5p/9/rlfKXyy0hvvuo3k1ebXeT4Eg190fIiVRUyElikA/bdzvzvSIZxgRMNtvUPD7iocJd8eD6STROumM6bXq8FsNn9/vmMlhpU7rxblmUdYSQIWzg/n5YlCf5g6HoBkbC70iUGPCPGZGLtG6r/wsZB5MhUF/unDeNwuQ//5fFRbsJ4y8rI2mecC5s6/7+cX89UQPJoiPqvWEBeYiTvizzj5owBI7Jnlj/CMiHLz3bj0D9iiwdDngY33E3beWxhhFZkHhvs5RO8JmXosLaXbvJ9JEoFGIEes+F10c1Y3jtgC8u278/546Jf7wdi6vGgflmIENIjGGNm4Luso4em0wkt39Qmt9AQGdumRQn0QlHe/JZMXMOK+egVPx3NR+i2eNIBFsz9jhiZ/dEYGeCWbFFg62K17S61G0saHAdhRgJ30by/j9EBhLImjU4xCst1TNlLAUvrpfvw6L3ZqVB3JvFWrvgWQEbiFdTqvKspaMNTWR8RLaGQmM3ex9BRl+GXpq833EJPvcbTJ2gk6VCuKpBgXutU3GRT38nfa0PO1yrzvKraCY2g/IJ5mEo7ALiinvkX14NmFDlvrzwNiiM7LMcYyhsl167OxgL317yBdxtTCrl6kwJ4jPdbedIapT3WXIM02mYpdkZTtaff9spuFmAda9ejCu0jsSiOz68uIlcmf6Rm473Pp3TGRHoEeIwSHF4ohMRCjR8v6TEDQM2GNnidvYpZu+TUp+9Ow/AI215+GZxxaYzjZS38uhGekrjY/3p8gaQxL8FM1wSepv+x+wfp5h/17NyJq3NhEgic1nHcRwVE8J/roxPJYe7Omep4a+9VQSWDZxGdcdI9EkJJHk3wAtRBSMKDzWU2TPI2VU42sXWvCBUyoAoD0EAGND8FQI3V3GyGb5DW4sukuS9c7ukF+vugw8PHy3q4dnGQRglUD98XhEpxdnWH/Vbo/yHw67w/k7jDUbOwobq0unGpqhiys0qtulcybZyN9Rd6D/YkclcFKzSGUdRGXYyQHO0BA1gLHphXPCh/evlopUJh/vizz7mz9dCGkJ/xEeu+lOyXRQM6HQXkfnf0AMITneYaq95y4GCToPjdiVIR0pCE8YTsZX6sj4wTeX3uHdyDHdICP3F08x+FPBKW6YowYAXvdh3HCcD5FXmbVNRXgIz00j2k8ps422qT000iolWV+sLQgwPlpPqcKUhvYsFcPCjGCFTDQfAvv/qhdZ5LcGx4bb+lRWlI4yDQa7Dx6bqnmpcBADqNEv3weg0E6skuyPfmshCo33YeJ/Hy4QhDDqHw8KsbUIlpaP7CbBQxvSzEVVsFqT36/V1C8qV9tAz3NXYx7sW10mx/Ou7Ff68dkddTzSc2fIm7LoKCt7+f/nB8Toy8bKpU+DybTAEGUB6Kl7sbwJlFNXrNXhh7CGvgztPH7pzlYK72LufbaVRmISxrtz7BXemLndfbWOzgjMXjC/XA4jZFanBh10JNouG7RqziUgD1H1EAex1kdrfyKbZCBZFELsW6z2p8JspoiLWK/5+e78/FpUBNmYBWaS+CfzqIaSuv5Ol6D2hgFmfvz6xbMJwO91kYdTF4+Oi40QnrVFtqzMaE2qUQUadnMcnw8zc+HVC1sTFiAoAOoJskFMY7DckoxXvkSNiEHzS2UTNzrTbjoI5Mxio1QtCv472VAxoLomKWXDlUHN7OAU+UDjZmhXlkZk7bLzW/j7ivWj3GrkaGQGYs8kw+qELCxnynsJBxmLTed84LP02nBev71vMA+ERO11IjlwxojKULNxume+uF1vdLFTK3cfiJWzNCfT+MDIrHSTC2ZNTX71BbTImfs+O81D0/OGfloX9YB6775Lm0a//jDfOq3YVJpTwZxVoNxzJeSvXUmk0ECqP1SvV57UMB/OG/H+3maE0jr5blofoqYkiL3tvPJ8q7QlVWHYGM/f52G3tuQeNk0XuSv/TMCcJI9jg+n5WIB2guJ0a/TTU4m1x4y860kKedin1o7rpBoX4YbmMh9VNWwIOQtguYcDYHhPapeuKv3dIjcn8ZDtCzF9+hiUq5gQC3liE2n4yq2AwK1LKiQJRARcP4iXt2eeAyh3WYOFTvxLcUbLtvh8zwuK1Tdeg+suqbbV3AShfB6K3rq1CIMJbsXktcnqJkIz+KdIGOjA1lowkIyJnDC9UekSkGT2q/CV4Lvri2SpURW2McF+kBMiokLTpedfUW9+WYZrInUHEw+4vyKy8EW+wkR87NJ5AimsDLw0UCNjJVxNy+WvPXarPNurUjeceNWv2ZmrsEhXQj1CYiOMtD298gY0AwaCbnM2eBp+EYEUBLa/n8jKithdznHWnrndW6HlSyxj3BpASTvYAOgB9x4A1JZH8fj8zkqKSRkQl4r9YmVHalxCsxnGBe3cYAqku4gP8uzstSIN/oCs/Z5a4xsFRk5cfrgCVZVW3JSfp/OjyOHn4d8hUJ10sJH4UUAdBrYNy79swVQKIJqcJVX4iKkJYcBuwsSB5N4FyrzjBAkG87H4UCLTrwFZcjWiPMWcgwzsuWfTlHvcFk8/cpkm5ewvcJAgHRTQx7sj24tapawvn6itLzLIHxX5tZ5e07IgrkBECHkJriNgRkSCxXh5I55IXFT7Qp+qFqf2uFW7/MACSy4ifUhhWrqsmgKIcGJKwBvUlLNLhgvIzl8vWg0BdTBnHAl/pVQFt7udiuADxWzBqVrhDjBLJiBlRgUgxEuApFKg014opCQsDI9Bw7phPKsLa5lHCXVXOOUvfmon/Y2TmM6fqcRPdfaEJoTm+m8+vs0pNHSlXenVslpWOakwe+VMHFGSj81n4wt5LcwulBea8HkdJe2BQjG0/k4XcdnV15n7yw5Scb78z6lrrKYkpxQlxQwfjdo3iLrzqWmFQt1JIXDekLGJRUw8LQVJ+1SoPuK1mHBwdg79lepT5pWlXl3SMQgf4eya/c2XxlMFjqnPs3kUXztSz/NJ5MxwbH2FOjBT2ebJrF4+dzvH3sboSgMrfCe86QzOO7gBD5AtrQIzZysfOxldxsiDOgVfAzZVD04tICWf4X4+ByD/dl+nflcsXT7wGpM1PMSzLm5CMnyWdWKGnY8ZMtqGJAexsFaGNzukR6hDCjGspwPpwTSxJywzHsMboGkfsdrLfj8sEpGwPJhGJ6hwT/NvY9Y0ljaysunCTHkqsM8wlBmkzRC6lKbpWOAFbbAjzzuTJ6G9zfHrM1IL/Ev2/5anoqCVSCGtWhBkhjiUoKJtQopOfKTW3pkAI/Ipfzce3I1cKx0ensmlhQLwbGhKSSmmrZXRoMSEtcf+3s4X4zJpvUxIFUieJYQeEFLQTLDSkniHJbwy24umIqxFe+m81rncVG+bdLDDza3t7BvIlJoFUPY2BTVLGX7EaTJoWkDEj2hGUP+z7oJCsLcwwpwHKyVWj+q9lpdpKfEsM/jwWRwNaXND6hTrgFr7g+ofHIiwwEfuMb5GxIA2lTOIcPzPC1EvYZ0LiYkJhDklvaofYF81FWQQKaBdLCqpiaNkooFQPS9748rDNthWl+eYAXKycd0eIw+UGpc+HZMqg3UWDxL/7KlTDDL9OrSp6J0XrOIGAp/RMJScGIEu7uRG5p0wUHk/ePUIyFlWoPqyJiCZzfCIEJBREPckVqN2R7BAwUdnGJeG9IAIn1Nn4OAxYsFQDWDigthnMs4NDQeEdFs3u4p8PNREHRKVl7yCydwJKdgOaxBH5hD5E1MI2h8wlIEdIhzHO5CzLyPDvPR/XapU9grDgeK/vJhPCJLdZqQwYUoIjkxiePpAZvCCuern0hebP4y3HM1KcMag/lNz8hIzp5qmB5OPkYDcyg50OQtKZOvhBn4G8hIJ8ou04COSrYoe0mo5M5NajGrXiO1PIWIBhbpyAD2RDYUpS58Br5Y2QtLTEEUx/uJYtr9J2Tet69KeUKP+NhlPowUlWIDlqBfkTWiJE0P683ZKWgprPF5QFxjEpTjfGbpSl6iwgN38P+q+TNmFcScsjWpMM4VSMsZRP4eYPjvXy97mCBjf8ihxXpveEbqnKPoafOTAy2eYOEnksJyMyFeV7zd8dwKP9eUYuBJQb4iWnF96GGVfDo8DliYRBySHKqQB5lOF9YJsqqwng155w0sjY97jOQw+PbTMkBWjfTeK+zj/exc1VRmho3ik/W9V3wYcFkjnLyRGqo+hhHCnU36zCpxvPLOcZac8seW4e5yPBr6Vsgpt25NjypREM1RsyJUitCMvsz7pEKeTACR+ff8GUVd7s1JUHoOTSdbY4hRd49P5ZCB4DMdQy5qpGUbzmSp5ZzU83W1dMji22Moh/39VQKCC/pTfr0iLFANBK83U8XcRnWTx3YqCvPcwk0surE5sJQF+7iCCHFcnm2/W0Hq4Dx1bq1U4zAmzI4DbVKYKogaptelnNniVDt+vZVKE2GyKDbFhhqskt0YBRM/okVmvQ+l+B9LKpEA8vu4O51xNq5oO6Y9cn/UwqBF3lz4Zejx//2AjJY++ADNN7TRIVr6L1iheBhebcCnJ3LSh1s1kXMT8y1cF7x0cehWqPDimw34qLBcD8ykCZyvl3m3JD50vz9dCCG5Iqc6N/305gZWi8Wof9Fi2aV7Num90VMXs8huAvRtKzhrbdyEvkzfbZuo4Fa4/OuM4i6rzcdL0YXv7FTFvoLx0P5eG35mWxPs7JtBCh+iNDEPqy3xmq88ACEEDPOXy1FknjcIxTRpr7zg1Ugdf11hpZjpLWwr4f766xZWK6DeADcy1M319HosV0m9gRQWoiHqrQFCgjkvp5dxF/RGx/4Yn/RClvxQ7owrJp2Mox8MrYvaCxSpSNuRgo+VGcKnY7gyZQ6pXaHlAnUB1XHSxzPsfD45t+LUpPCsTXdgbv8w7x8Qzj2vHfucoOQKfbYWhMqvjJE6URZFyfwZm/7Dr9YAESuFgPZz9MqzNwFPchfXrMoE2opqUe0tNboE8YCkwpxe5Cw1fArPh3mKyamyYLERSA9ppY5qxkpbBNWcyD1dft8HmUAahvSfx35dmcWp7l75jKIIagV0oIoOqxqVoQCMC6pyxEF7RVG/+SE6DbpYerPK7MKqAPxmvEO6SHBhqCzXxdCMmkNuIgQeSJg9n1CfRo2aUMvJV5mbZyAuPDEFLDG11BSkCeRnbalJjxmRhJ4GfDlKG6OO89aI1PAfnb+gEk1iiW0lCzDzz0DtiHrpbZqLjyjpOOKY86gjPXyEKKj0sqqx5yMlnFebI6BhmyxU2uGqzp74YUTzFOthQLWkBe9HSukSMYAqmUiFGXU1t4aaCpA+I9Q8klMNUckr0KplhjynQjYjpCn0gqlleZdSWwVWW8oVkDCNACHR8ogjdYWp2U9LCWjQwvNyBaKqt8vzRcMzwGB5xcnT8KaKat3SMtv2zx8nyMdvUDlHk99i/AIvvUYY1StgqvB5ScDe3qo4aeQVNOyIx0eDDl9Y80KQZ4KFSxp+3PF55Y7uOIJOnrWldxz5lFYAVndpoU+71mCa/2U4HkY4vdcebz5fC/+rsiAqoYhjZhmPq0qqIZFMDp6IaNH4+HWY0qIZQaBuLS1sIRigk6FuQ9E6KXrJoeihiCWdHzBGI0AuJuWKUxM6phR7icQ5E0O8MYFrmoPfcJ6rJRYFZgrVD4iLSyoJrABonNhCIC65ttLCy3dlh8vZvNFyudWCEnUjCqbP57skmUkVBzo5+Jl+LEwbC+KGKA5IrBUxcdwxn5KiunaOi5oxdxN9tMXkWjmHww1hbmktCDFKYJwoMWLDeGmpcVYvDr9kpFAHZz6c9/tLUjyypvOMohlrCXC9JhdBOoXGI51UoZpXubSRzGRHCULPqIdxIz/IF9lK3lpS7CWM+FfZS95sRmwm41/21MGh9R3pkzGMhxi840Qn2lNNMhLQ9j+el/NuMq7wnINGKo5IoKLmhSEvOeN1N1nnpoY4OM9Y7dHD4dhIaLt4Hu8NsfgBf6SqtTkZHiIAjpQnsjnsnxII1W2CfAOe3+RaFyBiKEpl3lPKSipHY9kXLKHoszSYRjHXynBDEiMV2WnkV2lh4Azz/uhNsZrtyAKj+lUiKc348/Dmz8sf1lFLmYYsNT52K4DAt77vkcZoVibrtxuOmZO8pUhM3KE/XeV3U+qwREkUmt4WIQV5kmAZBtfVZcjD67h2D0mT+suu3Yqyyagwsao33sjf+nrG8jTrhOro9nemNIM/d1o1SInxq7UQR0FUsDkiSjQFUXfUxuC957dxPvbq1UbUfcTWFBmd0AqsFWdstqT0hhb38/5tOyIQYcYZnqak7GuTRdkxmVTwjzPydsY50se8qjzzbi31ml7DhO283gTFH7fRoWu/C+kgCLWbD4iYPMZiZJxl2HCCOvEEA6CIbwK8pHp4y82SJCRY2rsBAnsrykwgEt7vWPY3+NjNWPEyAAqanAMVbLBbvAkWFLFH2RMZ5yLS8/vdObKFMA5UxJerX9e+BnaEkQ/3y4gssf4ax3UqeKu0teaFRKAjR/1+/DzuoyopNT3w9R0/KyPI4IB5P+73pniP3OpBWlfH9eUtMXnnnkdLTdXiO964HfXGblmKtYYEeHlaZSdJbQgX/BzJd1NC4vw5hK6SKUxCAl2mhS2thIT/fHs3oxoPUlINQiquN7xTRMm5iVBj/k08lUuSAm3y27IvjMqvyJelaKWQiqfushDJn1sdqdZ6n1/Jrbkq+NmEXaSpCUlvS6l7B1vzk2Gsua/vJCzcDiDOPpSag5l8nJe02lwuIQBayFXrBQSY4/pxCxex1mUTDDiF8/X/kjfV8ilYaouNrGEHHB94nQ+WiZiGy4PBeBGMELJfVCXq2EEVyVsuwPA0kvvHVmAQ8dQXP5ao1ojqOF0ck2sl7ZYrU7Q1n4xY3bZbZJpHVPMw7031PG8RrX3hwEjOkso9kbOpNlauUm5veUZD/hcY/ed1MrDWXy58Vl/yBoqMe76Rz5iH6NXGB5JFUOUzzv7zt1XKWSMG7ybVTKlEcD9NpM5eFQWiIniFMxXJ25tAPbl+pxEtuqhpUVbOwHR8ROWSOYZKRkVUPAhNuIUkghr2Oe4pRtm7ETXdSkKjNc4yAtrNexjvJ7+3BdSY0ih6lkUEVHAU2nmOazoEP5JNgNc1PfNryPcIA3uzAvMy15rbWaPickMr8Qr+K2mtK7iwzn8PTx969X6tVfvvwHEPxbrzasT6t/Af5pc1XrbT78MxBX//MhqbqsSn+vq1paq4bFStqOAyFGkIgkkEklVENGYlIFDhDDruQ9I+J2tn4VduXlCiTF4/gXnIalRpLDg51LKsVv2Awt7rp6dxlXzamUgtF/8eUG5DK2bpNYlJ3tjlkiJRq4OjptUWIDV7fvC//pgcCL7emr9bxb6m4aIK2JI9Yn6GJVixVY7WL3TGlRMrTt5o5EcUZxj3r7ZRpO3YNsgUhroLg81uCClfRZhvK9BApfn4OM4xGfLa1x1oOw4V7Q+Pl7XXZ/0nQnI+w5GJkCRBdcJPtD6c7z9Hy/cXSEAohBQGcXVaXr1pjf30dEmgVjvOSlmdMhRFnHeokLgkYTibSS+ym38CouKq9lTGLjJm71i68cd5HBUaGD9fbOk1F4sUUuWwMHgU3bVYUXMddGyJqaDrEiQTU/Wl808qehvpcfgPFGkYaj7WZQwEpM4ESiyRcbeX6GnrdaCJLGPVGhcxDh/87TMCwJYVhPdOUfpcDA2SD6h88+PxsX/BRVbGwqbp+eWqXw7aHoqdG9rGO/Hb0oePB1pyriOtwlB3sWBs1qqcGOmJ4YBzxIpq6lPi+slk+ioMMUVboGz8vEsqQchdT2wUBYeJ1DWFeSHRCR4W03gdKqOvRrKh8hbb0eSt1yHUxlfEidTk88CFDrANb62QoqmRla9aFgEdGXqWCTGqukMrTiGEHUSekEMDMRr5YRzIeu7DcypSMMnY0Yo73RIjLBwFlvudTcV1HJUjT7IjBWJs3xGR24aa1mzLuSn8JC9SJC+57SGIub/XdCVND4lM/GrLjlo+UCUGcyPMa03X3LSl/t22GzIm7A1vhowmMy53XhW2bUqlxWZYLD0L11oWmAUlmA8jgvbpFtrd2kXmA4pgDyslYj1crJNRTPCeyi+uC/vVfHmVaIZVkWzTgooUPQxfoX2u/WpBXdDyAvZdXATg9DXJE9AgcH4Lno1ZP+ASvw13R0p/6Ff6TxZqT5YyJAED9Wze4XLNhcqOqCnOe2tDimkkx1V9KGw9+5op4YKYzLsUC7NVKDobcsEUwid5HsoQ1tay+S6SN1QniysHpqn1WilLshg40C5isH1fboJogUguQyOVEM1SKajm6TjdnZekrLx3gVXsFg/U+JdvvyDrjdOulF4kI1F/XKy82VoYitAh+3U/GwMks9m21qfLblCLESAB+ppwpVgoavEirvBI+h+R+r4sNvnM+YqRIX0r0MMUeV62KX0b0vjDHUYRgBjdGQXVphWGp7/yl4rU6YmDN/58YX+3ZQ5ZbN4yh5Ku80K6yGk2Ocm+ZlTtm3d8SkQQneLwB9xAaVETveVOwvICDIc50jcf7YcUIbeoDJfyBHrYgfqXp3RstcJS6dNwAjU+7NPw0m/9FZuaWu2II8bcZLOMcYQg4n47+TRvVVP4EO/CTQcWUJCFAPkv+4sB8FYvoybZWQTfUIvI921y7afWj+/YWdjxlgmIigLMkfm7xkjcAaScir2flcVAWEO+9xXE5211On9kBI0gKttwohj7IIL6gvOFFM1v1N8ZIQ317fmI5PLTKQGR/CB3CDaSPGBRuI4T5TOncUpdhiFBozQLngp9kNixI66dOg2dL1jRGXrA4ce+P8OlE842X8hfdmFnuAPZeiFySDEMyfxQ9wHYW6h/Yw8RitBeZtiH56cEUnNucpBC7SrHcgMEmKv3sHDb6LnTpCsFtl7QoCzlUx8TdkQWyn1Fx0jsiPiZq6jsqWDQli73/UCXj2hpGeUPEqmiaZM30b8NewOMIVzOX6eSaxkyK+3xxXELZbgcbaQS13xlo3STiHtY3+9ehrs7W2HNZbG2VrlqnRRE1FSw5E0oBl77klmRHmziHqrn7uLLRCm7cD7oWNKYKrMSqNYGMA+nNaYO1XX8fTsWhfUDrRcXTD6uTz46H1sxQjAv6Mz+hHjyT+QbH30CrdbMiGFdZqzAe/44PyDUdNxFV6sTuVVvJ8xWCOh6uKZ5Pu/SEmZaQ1py/nAamcVWs38AE4jWExA7CchfxWH+NcdTR1RN2ufjsE9KwvMxX+gm0Jv9AqIRRChhH62evhhfXngza0RRChjqEVK6TLihNVPbfyUeADP9YGzfURoPBCR7QVTntHpAy06AQI7+/QZ3jy93EjJUW5/BFylzopyJTPWImJtaBM93JOeM0ft7K3w4l9xKmPQDE70mb3xBsDpkEUR6OCUpOOXU33BdXwsBZJ2l8tlLAghRI03Ixg6AWttHDIY1w3n2ES9ZiJCGcq5Qu/y47pLnGaloivX47iylzm1mRhUKc9TpYcABzuf9eI+8Xatd+LtSNQXdQjCL77iWY9Cgo6JQSSGsyqsaAcNyAGk8CYKGqGEJvlFNKSIK3Haxp3PqpIHaZe2XqL9pORKXnhjrWVI68njPTMNyQCSulPj8RVvGKtQrVqTgT2OpqVggkiFMOYkQX1VnSbpUxNQkVB3v6IbQBBNvbQw3eUZQA0PJTPbQK6OwuXypFu4XUbihi9jMHd3YHirpNC563e0pATHlWyTWfabclKvKn85mhds1BWb9HgLcLpxEXvbporXKpfpUR4WqBbMCkb7S+mtn2nQx4rB+v+gsqsQrDgEXry5WGT5Ays2fsZsQzR8y+bS0RpbFu0j0wyKKEjRxpe1zrEnps6/aLN4HFOnlKpVRgtzC/NDkswTnGh/cEhA1eW4+Uxn3qU9RDfs2aj6/eLTNqsc/4X4AeBcQAa3jJjMkW4rqEHOgiuXl2DWMOelNWREEfkb3REAF46ptdTqruLpronxkTWuosnh1bMokagzlX6aYFKIqbudn0VnSXFpdkfvA9iYkVwYA9Dtkbj3Pty48D4m5kZwuzEEpQzIGrK5KciYDyiIqj6DMEhOxGtxt6sdIPoOuZNpdhtWFriFkz6Xjg/Ci4XEYEvpQzFX9rUmf4N8fEbB+78UV77sKoRAu2YI13ctGSVn7kwWEUiqdDzCOgI5vNdsO03kZVhi9K/fKBFhjVb39gjjdkAtVM2MOZbMaQ4pbRJb+CzUecqFqFr1D7lptyQtELX2BTBDSSzhSHbsn5K51lryku/RgR52Vm7uk9JUdypySnHq64kZI9QwNqqKdVQomXZBwgss6JuX8XjvqXOCAkbzmcMnnITLWkIpeO2viqfHdb+/vKTdML9lqfbCPT8GuLHXLMW8o2Bw0nnjFA2au5jhDSt7trHRYU33fcdg/zLMNK/JBpOHG4UBfZEwP2+G8uk5aBIHOrSWsGha9d8sZ5YxtllvTeCeVZGwYKb8u+M51c9m3suyOFTYf4F3aV5SEOC9PKYJC6ooqXqlmV3NBRviX/RjszcoU/a6vsjWfIIPeSEbTgwXUybV5qaW/huHpmxkl1PvJmDP9egpFNJJuIcgLgGm4iq9jz0MwnicdQ3wgpy8slxsXqJBHpOF73NTEFWCowYD83/7ukkR6Nv5qxmrtAKhL9tFtR75ULokOjfU8nA9uiaicPaWIHFE1t+V8yVoCXus6WV9Uqve80GhFUcLJpTqsr8hdZ1lnEHD6DiSOc4d8GYkixoXSmJnVBePcby8DssPPxwTSFLEEXJNZBbLGUvv+glJYsoINpit8yoBeX9VZlBR3HUhZk2LhyxCZvfPZG5G+oWo0iLcDL4mA1msWnpeIv89joHTDojuNoVpRdKCJKxqzLJFjIhSwn7CG9epnSkF6Gn0egYVFTJclMCzXXxDH+XUMwTartymk6FJYTjlP+357Pv4JJdHwt5PP4tL1I1Gvwa3Plv78Fv5nBKbjDp8b6Cy4z7kHBg1p4BE8ARHHV69fi8W/3w3TUNqPW81oGYZKxQHTDFUiP8JYiJpNPnUv+HZXAQddFmFYtgPCsgPIsHcToRAd9wFIxbNQjvvG56dBTcwtY5XiVa9jK+m3X7dRhUscWxYvQxM18jRxuocv4FsPixA8p79YgVkhUC8WWYOvoJsk5uEW/FfSj1+Da5ZzE0r5BXiDMg+4mXT+ER5S3APlPf+qbpqVH5d9lwrfsQ1MOQ6oMPL/X21Qxa9pxnnY0/VS2sLqgg3WARs5Vlv+YyVi08RHJDWfqUpE2o65US7z8NhebKijW6iHI5ae1fVyH97so5wNf3ace3Kk0lbeVVtzcKrT+6TBw+pAnPMtbAhJi+SghQmi44xIJ2klgbzgtlGlAb1fsySKTqawJYgTDZcBoPSCgESq94DYWdR0NJcgOG+oktDBQFzRATt+8QWLWffvqDOSf07VDQMt3wRweERp2cQ227Atn/hwrtf0ZbY7NUW6gR8H50/lsyfFBEIJCIG4IeJdz3577yuqOLqjYCGh5vM1kFN6KtwZVCfSVuMqfPBmVsoMFgbTUWrLfpjuLknJ4JqfBRtOzCswWO9GuEaXY6RXxiCZrE2j8XeNmWxM7HsIOgiAN5YQc3MCu5MrPr1r0ze6kI5hKE41LEmVcPIrs+uTlkrB9zgGFFJ7UTINNxf1z0MCaiT5lFNCyN0CoS2iqCo3ijYOyx1Wu+RTthptXzTqK4zUVJBreDn0oRJywWsJlCWn3UVK5JQh+BZ5U0lKWct5fBLC7tgFFBGs31GRDu67jpa40B1XMCLTsR0mOHAuS4iSUUBbazC9KgVs9w8QSivr7/7Pf+HH3xKDtUdfzPHkDYemBPKGCnLNXHHG3/gmOaCNRio4S0yTPS+7wTufeCmw95ij/J1s0UBeUHHN7TbUZRc5seKlQFqjbZqk/x3S4UI8V6F1qgqtVVSbVUehtv0+LZnIbrCCt1qtRaQiPQdtUK3YSB/8QgWb62lj53xhZADVdGniA7aAtRK3ISJZEgYjeUMXUpGzOFpYWdiVbV1n6RcQN59fkAd6/6FPrL58lSSFSpdSoam6CXo/PP/7MLraYz9sdz7CUTSrKlxKVfI1DoXpHhlz5/MdhxVbDOd1iagiV4LarUPX0FFVDNgcH78hqB5PtoF6XWLA8mjbgKTV3m4hLVCQtEBpSjC8fx0Gs0SdnAbsVinMsmgpqA/pBye6esLTk8RESS4FnYDYvXbLgxnhXpyFLnny1zXxFJOrlp3JlSV2dEPPcqA735S6ldLKtffQm+WDf/zT/qlXPS4kojFLcHwhBy1rO9oUyw9G2C8phLJAOOmYeIRLzjX84zeQglD2yl5rTH72gnZP2emtnhEgddrI4ZhiGs3OIONupQwpoDoqqouE94M37sYYalt30LyHokAh0iHsdwiZPSLi1T5qnrKwKQ/PrGt8JmoSUfbVsArGqtklWK/I6W5LFLnF4ojRWFzWmKSMQrz1Zv46ujcKWplc4RBEdeUFylO5RELFYQ4R6AQ4kSFnjaOuaQpmx2bYzL6Sr4mhAlRyP4cqwi11jYKQxX5vh7qRGz5O8F8nGLpSFwpUnnN+kOsS4YPKKj/jQnaNgOL8mYZXMJvnai6/F6lRewuBkWSkC+RskGh0k3SF5ZsN1VS+TJ+HUNNQExBWOR2BHKaF/mU6g5tH5hyPdUkkaioJ5bKvoejOLcl/RneTkIhGZUw6NRopLxFQVOBzuSMDt+jMvtDxVe2TCCH334zce1MLKlxLIPVDIy3teUhRMMUF+ydvQ8duXQqwTb6dPH8z3ZnkqeMVAZ7PtpYeqgiuZ30a0zLNjWpprWR3OwNo+YqlE2qGKvvU2JzSu+VlL9px7eg6lc+4liJYldVLpWEgkvNWsArgQfj3j3jHU383z2+RPoYwzTefkAifvpSllUZ8dFKa4hb+E7bJdANdpehX3g5zza13uxRdvYL+dEaA9A102vPK9hzJSAjv3CEo6/Tql6u4UPBGaZwVapMGXvl0jegM8K67DX/l232mk4fXr8Bf+XgtOx3xzvae7oxdds/BHSvxTq3Ug3NWqGiw+N/Cx3je7YxNHvuzlQu065SYLpQeKdBReYHukFAfpAoVuQPEEeQz3YGaYlpfS7drJJawNKB68w8gcLNfEACKmp1jbPvPSQbm8WaDQoP+/vPzgMqtU2o8/V1Mu/ntsqOrHVz231WiuZFknInNq83V5x/gHWXE7O/HYcWygtEoM0xrjeH4ylfwPmuujiXAfAMFFRFAahg/3iAS8gFZBrFOqdrmQ9Ytd0NKPd9u4e/7u6TQKQ+elteq1XSj10J4fC6m1H24n13ObJ9gUwRrU3zp/wV45G1ugJ8AAA=="

def _test_text():
    if TEST_FILE.strip():                                   # 1) the path you wrote in TEST_FILE
        p = Path(TEST_FILE.strip().strip('"')).expanduser()
        if p.is_dir():
            p = p / "pokemon_data.csv"
        if not p.exists():
            raise FileNotFoundError(f"TEST_FILE = {TEST_FILE!r} – there is no such file. Check the path (drive letter, folders, .csv).")
        print("🧪 TEST DATA read from:", p)
        return p.read_text(encoding="utf-8")
    for p in [Path("pokemon_data.csv"), Path.cwd().parent / "pokemon_data.csv",      # 2) a copy next to this notebook
              PROJECT / "pokemon_data.csv", PROJECT / "data" / "raw" / "pokemon_data.csv"]:
        if p.exists():
            print("🧪 TEST DATA read from:", p)
            return p.read_text(encoding="utf-8")
    import base64, gzip                                                              # 3) the copy built into this cell
    return gzip.decompress(base64.b64decode(_BUILT_IN_TEST)).decode("utf-8")       #    (works without internet)

def _int(text):
    text = (text or "").strip()
    return int(text) if text else None

TEST = []
try:
    for _r in csv.DictReader(io.StringIO(_test_text())):
        TEST.append({
            "Name": _r["Name"], "Type 1": _r["Type 1"], "Type 2": _r["Type 2"] or None,
            "HP": _int(_r["HP"]), "Attack": _int(_r["Attack"]), "Defense": _int(_r["Defense"]),
            "Speed": _int(_r["Speed"]), "Generation": _int(_r["Generation"]),
            "Legendary": _r["Legendary"].strip().lower() == "true",
        })
except Exception as e:
    print("❌ Could not load the TEST DATA:", e)
    print("   Check the path in TEST_FILE at the top of this cell (or leave it empty), then run this cell again.")


# ---------------------------------------------------------------- 🧪 DIRTY TEST DATA (for part 2)
# 15 Pokémon rows EXACTLY as they come from a CSV file: every value is text, and some are broken.
DIRTY_RAW = [
    {"Name": "Bulbasaur",  "Type 1": "Grass",    "HP": "45",  "Attack": "49",        "Defense": "49",  "Speed": "45",   "Legendary": "False"},
    {"Name": "Ivysaur",    "Type 1": "Grass",    "HP": "60",  "Attack": "62",        "Defense": "63",  "Speed": "60",   "Legendary": "no"},
    {"Name": "Venusaur",   "Type 1": "Grass",    "HP": "80",  "Attack": "82",        "Defense": "83",  "Speed": "80",   "Legendary": "0"},
    {"Name": "Charmander", "Type 1": "Fire",     "HP": "39",  "Attack": "fifty-two", "Defense": "43",  "Speed": "65",   "Legendary": "False"},
    {"Name": "Charmeleon", "Type 1": "Fire",     "HP": "",    "Attack": "64",        "Defense": "58",  "Speed": "80",   "Legendary": "FALSE"},
    {"Name": "Charizard",  "Type 1": "Fire",     "HP": "78",  "Attack": "84",        "Defense": "78",  "Speed": "1 00", "Legendary": "f"},
    {"Name": "Squirtle",   "Type 1": "Water",    "HP": "44",  "Attack": "48",        "Defense": "65",  "Speed": "43",   "Legendary": "maybe"},
    {"Name": "Wartortle",  "Type 1": "Water",    "HP": "59",  "Attack": "63",        "Defense": "-80", "Speed": "58",   "Legendary": "False"},
    {"Name": "Blastoise",  "Type 1": "Water",    "HP": "79",  "Attack": "83",        "Defense": "100", "Speed": "78",   "Legendary": "False"},
    {"Name": "Pikachu",    "Type 1": "Electric", "HP": "35",  "Attack": "55",        "Defense": "40",  "Speed": "90",   "Legendary": "False"},
    {"Name": "",           "Type 1": "Electric", "HP": "60",  "Attack": "90",        "Defense": "55",  "Speed": "110",  "Legendary": "False"},
    {"Name": "Mewtwo",     "Type 1": "Psychic",  "HP": "106", "Attack": "110",       "Defense": "90",  "Speed": "130",  "Legendary": "True"},
    {"Name": "MissingNo",  "Type 1": "",         "HP": "NA",  "Attack": "nan",       "Defense": "?",   "Speed": "29",   "Legendary": "yes"},
    {"Name": "Snorlax",    "Type 1": "Normal",   "HP": "160", "Attack": "110",       "Defense": "65",  "Speed": "30",   "Legendary": "False"},
    {"Name": "Gyarados",   "Type 1": "Water",    "HP": "95",  "Attack": "125",       "Defense": "79",  "Speed": "81,0", "Legendary": "False"},
]
NUMBER_CASES = {"45": 45.0, "1,5": 1.5, "1 299,99 zł": 1299.99, "12,5%": 12.5, "-3": -3.0, "abc": None,
                "": None, "NA": None, "nan": None, None: None, 7: 7.0}
BOOL_CASES = {"True": True, "yes": True, "1": True, "T": True, "no": False, "FALSE": False, "0": False,
              "maybe": None, "": None, None: None}
REQ_TEST = ["Name", "Type 1", "HP", "Attack"]       # rules used on the dirty test data
NN_TEST = ["Defense", "Speed"]
REPORT_TEST = {"Attack is missing": 1, "HP is missing": 1, "Defense is negative": 1, "Name is missing": 1, "Type 1 is missing": 1}

# ---------------------------------------------------------------- helpers
def fresh(module):
    """Import the NEWEST version of src/<module>.py from YOUR project."""
    for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
        del sys.modules[name]
    if str(PROJECT) in sys.path:
        sys.path.remove(str(PROJECT))
    sys.path.insert(0, str(PROJECT))
    importlib.invalidate_caches()
    return importlib.import_module("src." + module)

class ProjectNotReady(Exception):
    pass

def require_project():
    if not PROJECT_OK:
        raise ProjectNotReady(
            "⛔ This part works on YOUR project, and your project is not ready yet.\n"
            "Fix these points first:\n  • " + "\n  • ".join(PROBLEMS) +
            "\nThen run the Setup cell (part 1) again. Parts 2 and 3 work anyway (TEST DATA).")

RESULTS = {}

def check(label, test, hint="", details=None):
    try:
        ok, err = bool(test()), None
    except Exception as e:
        ok, err = False, e
    RESULTS[label] = ok
    if ok:
        print("✅", label)
        return True
    print("❌", label)
    for d in details or []:
        print("   •", d)
    if hint:
        print("   →", hint)
    if isinstance(err, NameError):
        print("   (Python says:", str(err) + " → did you run your cell above with ▶? Is the name spelled exactly like this?)")
    elif err is not None:
        print("   (Python says:", type(err).__name__ + ":", str(err) + ")")
    return False

def summary(prefix):
    labels = [l for l in RESULTS if l.startswith(prefix)]
    good = sum(RESULTS[l] for l in labels)
    print(f"\n{good}/{len(labels)} OK" + ("  🎉" if good == len(labels) else "  → fix the ❌ above and run this check again"))

def safe_test(test, *args):
    """Run one of the test_... functions; an error in YOUR code becomes one readable point."""
    try:
        return test(*args)
    except Exception as e:
        tip = " → there is still a ... in your class" if "ellipsis" in str(e) else ""
        return [f"your code stops with {type(e).__name__}: {e}{tip}"]

def close(a, b):
    return a is not None and b is not None and abs(a - b) < 0.011

def run_main():
    """Run the code cells of YOUR main.ipynb from top to bottom (like Run all). Returns (ok, message, namespace)."""
    nb = json.loads((PROJECT / "main.ipynb").read_text(encoding="utf-8"))
    cells = [c for c in nb.get("cells", []) if c.get("cell_type") == "code"]
    ns = {"__name__": "__main__"}
    try:
        ns["get_ipython"] = get_ipython
    except NameError:
        pass
    try:
        from IPython.core.inputtransformer2 import TransformerManager
        transform = TransformerManager().transform_cell
    except Exception:
        transform = lambda s: s
    old_cwd, old_path = os.getcwd(), list(sys.path)
    ok, message = True, f"{len(cells)} code cells ran without errors"
    try:
        os.chdir(PROJECT)
        for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
            del sys.modules[name]
        importlib.invalidate_caches()
        for i, c in enumerate(cells, 1):
            src = "".join(c.get("source", []))
            if not src.strip():
                continue
            try:
                with contextlib.redirect_stdout(io.StringIO()):
                    exec(compile(transform(src), f"main.ipynb code cell {i}", "exec"), ns)
            except Exception as e:
                first = src.strip().splitlines()[0][:60]
                ok, message = False, f"code cell {i} (starts with: {first}) → {type(e).__name__}: {e}"
                break
    finally:
        os.chdir(old_cwd)
        sys.path[:] = old_path
        for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
            del sys.modules[name]
    return ok, message, ns

def main_code():
    nb = json.loads((PROJECT / "main.ipynb").read_text(encoding="utf-8"))
    if not nb.get("cells"):
        raise ValueError("main.ipynb has no cells")
    return "\n".join("".join(c.get("source", [])) for c in nb.get("cells", []) if c.get("cell_type") == "code")


# ---------- tests (used for the notebook AND for your files in src/)
def test_to_number(f):
    bad = []
    for text, exp in NUMBER_CASES.items():
        got = f(text)
        if got != exp:
            bad.append(f"to_number({text!r}) gives {got!r}, should be {exp!r}")
    return bad

def test_to_bool(f):
    return [f"to_bool({t!r}) gives {f(t)!r}, should be {e!r}" for t, e in BOOL_CASES.items() if f(t) is not e]

def _rows_for_tests(to_number, to_bool):
    return [{"Name": r["Name"] or None, "Type 1": r["Type 1"] or None, "HP": to_number(r["HP"]),
             "Attack": to_number(r["Attack"]), "Defense": to_number(r["Defense"]),
             "Speed": to_number(r["Speed"]), "Legendary": to_bool(r["Legendary"])} for r in DIRTY_RAW]

def _ref_to_number(text):
    if text is None:
        return None
    if isinstance(text, (int, float)):
        return float(text)
    text = str(text).strip().replace(" ", "").replace(",", ".").replace("%", "").replace("zł", "")
    if text.lower() in ("", "na", "n/a", "nan", "none", "null", "-", "?"):
        return None
    try:
        return float(text)
    except ValueError:
        return None

def _ref_to_bool(text):
    value = str(text).strip().lower()
    if value in ("true", "t", "yes", "y", "1"):
        return True
    if value in ("false", "f", "no", "n", "0"):
        return False
    return None

REF_ROWS = _rows_for_tests(_ref_to_number, _ref_to_bool)     # the dirty rows, correctly converted

def test_check_row(f):
    bad = []
    good = {"Name": "A", "Type 1": "B", "HP": 1.0, "Attack": 2.0, "Defense": 3.0, "Speed": 4.0}
    try:
        if f(good, REQ_TEST, NN_TEST) is not None:
            bad.append("for a good row check_row should just finish (return nothing)")
    except Exception as e:
        bad.append(f"a good row should pass, but check_row raised {type(e).__name__}: {e}")
    for row, msg in [(dict(good, HP=None), "HP is missing"), (dict(good, Defense=-1.0), "Defense is negative"),
                     (dict(good, Name=None, HP=None), "Name is missing")]:
        try:
            f(row, REQ_TEST, NN_TEST)
            bad.append(f"a row with problem '{msg}' should raise ValueError, but nothing happened")
        except ValueError as e:
            if str(e) != msg:
                bad.append(f"the message should be {msg!r}, is {str(e)!r}")
        except Exception as e:
            bad.append(f"should raise ValueError, raised {type(e).__name__}: {e}")
    try:
        f(dict(good, Speed=None), REQ_TEST, NN_TEST)
    except Exception as e:
        bad.append(f"Speed=None is not 'required', so it must pass – but check_row raised {type(e).__name__}: {e}")
    return bad

def test_valid_rows(f):
    bad = []
    report = {}
    gen = f(REF_ROWS, report, REQ_TEST, NN_TEST)
    if not hasattr(gen, "__next__"):
        return ["valid_rows must be a GENERATOR – use yield row (not return)"]
    kept = list(gen)
    if len(kept) != 10:
        bad.append(f"valid_rows should let through 10 rows, lets through {len(kept)}")
    if report != REPORT_TEST:
        bad.append(f"report should be {REPORT_TEST}, is {report}")
    return bad

def test_timed(deco):
    def plus(a, b=1):
        return a + b
    wrapped = deco(plus)
    out = io.StringIO()
    with contextlib.redirect_stdout(out):
        result = wrapped(2, b=3)
    bad = []
    if result != 5:
        bad.append(f"the decorated function must give back the result (got {result!r}) – return result")
    if "plus" not in out.getvalue() or " s" not in out.getvalue():
        bad.append(f"timed should print the function name and the time in seconds, printed {out.getvalue().strip()!r}")
    return bad

def test_clean(f):
    out = io.StringIO()
    with contextlib.redirect_stdout(out):
        res = f(REF_ROWS, REQ_TEST, NN_TEST)
    if not (isinstance(res, tuple) and len(res) == 2):
        return ["clean must return two things: return kept, report"]
    kept, report = res
    bad = []
    if not isinstance(kept, list) or len(kept) != 10:
        bad.append("kept must be a LIST of the 10 good rows (use list(valid_rows(...)))")
    if report != REPORT_TEST:
        bad.append(f"report should be {REPORT_TEST}, is {report}")
    if "clean" not in out.getvalue():
        bad.append("clean must be decorated with @timed (it should print 'clean took ... s')")
    return bad


# ---------------------------------------------------------------- check YOUR project
PROBLEMS, NOTES = [], []
MY_ROWS, MY_TYPES = None, {}

def _problem(where, text):
    PROBLEMS.append(f"[{where}] {text}")

if not PROJECT.exists():
    _problem("Lab 2, Setup", f"No project folder {PROJECT}. In Colab: run the Setup of Lab 2. On your computer: put this notebook inside your project folder or write its path in PROJECT_PATH (top of this cell).")
elif not (PROJECT / "src" / "loader.py").exists():
    _problem("Lab 2, Setup", "src/loader.py is missing. Open Lab 2 and run its Setup cell – it creates the missing files.")
else:
    # ---- Lab 2: loader.py
    try:
        _L = fresh("loader")
        _name = getattr(_L, "DATA_FILE_NAME", "")
        _raw = PROJECT / "data" / "raw"
        _files = sorted(p.name for p in _raw.glob("*") if p.name != "README.md") if _raw.exists() else []
        if _name == "my_data.csv":
            _problem("Lab 2, step 3.3", 'src/loader.py, PART B: DATA_FILE_NAME is still "my_data.csv" → write the name of YOUR file.')
        elif not (_raw / str(_name)).exists():
            _problem("Lab 2, step 3.1", f'DATA_FILE_NAME = "{_name}", but there is no such file in data/raw/. Files there: {_files}.')
        elif not getattr(_L, "COLUMN_TYPES", {}):
            _problem("Lab 2, step 3.3", "src/loader.py, PART B: COLUMN_TYPES is empty → write your columns and their types.")
        else:
            try:
                MY_ROWS = _L.load_project_data()
                MY_TYPES = dict(_L.COLUMN_TYPES)
            except Exception as e:
                _problem("Lab 2, step 3.4", f"load_project_data() does not work: {type(e).__name__}: {e}")
    except Exception as e:
        _problem("Lab 2", f"src/loader.py has an error and cannot be imported: {type(e).__name__}: {e}")

    # ---- Lab 4: stats.py (count_by, group_stat exist)
    try:
        _S = fresh("stats")
        _miss = [fn for fn in ("mean", "count_by", "group_stat") if not hasattr(_S, fn)]
        if _miss:
            _problem("Lab 4, part 4", f"src/stats.py has no {', '.join(_miss)} → finish Lab 4.")
    except Exception as e:
        _problem("Lab 4, step 4.6", f"src/stats.py cannot be imported: {type(e).__name__}: {e}")

    # ---- Lab 5: models.py (Dataset and the Analysis family work)
    try:
        _M = fresh("models")
        _miss = [c for c in ("Dataset", "Analysis", "CountBy", "GroupMean") if not hasattr(_M, c)]
        if _miss:
            _problem("Lab 5, step 4.1", f"src/models.py has no class {', '.join(_miss)} → paste your Lab 5 classes into src/models.py.")
        else:
            _d = _M.Dataset(TEST, "pokemon")
            if len(_d) != 800 or (_M.CountBy("Type 1").run(_d) or {}).get("Water") != 112:
                _problem("Lab 5, part 3", "the classes in src/models.py give wrong results → run Check 4.1 of Lab 5 and fix them.")
    except Exception as e:
        _problem("Lab 5, step 4.1", f"src/models.py does not work: {type(e).__name__}: {e}")

    # ---- cleaning.py (today's file)
    if not (PROJECT / "src" / "cleaning.py").exists():
        _problem("Lab 2, Setup", "src/cleaning.py is missing → run the Setup cell of Lab 2 again (it only adds missing files).")

    # ---- Lab 5: main.ipynb
    try:
        _code = main_code()
        if "src.models" not in _code:
            _problem("Lab 5, step 4.3", "main.ipynb does not use src.models yet → write your classes section in main.ipynb.")
    except FileNotFoundError:
        _problem("Lab 2, Setup", "main.ipynb is missing → run the Setup cell of Lab 2 again.")
    except Exception:
        _problem("Lab 2, Setup", "main.ipynb is empty or broken → rename it to main_old.ipynb and run the Setup cell of Lab 2 again.")

PROJECT_OK = not PROBLEMS

# ---------------------------------------------------------------- friendly error messages
def _friendly_error(shell, etype, evalue, tb, tb_offset=None):
    if etype is ProjectNotReady:
        print(evalue)
        return
    shell.showtraceback((etype, evalue, tb), tb_offset=tb_offset)
    msg = str(evalue)
    if "ellipsis" in msg:
        tip = "There is still a ... somewhere – replace it with your code."
    elif etype is NameError:
        tip = "Python does not know this name. Did you run the cell where it is created (▶)? Capital letters count."
    elif "NoneType" in msg or "returned non-string" in msg:
        tip = "Something gave None – does your method/function end with return ...?"
    elif etype is TypeError and "positional argument" in msg:
        tip = "Wrong number of values. Check that the first parameter in def is self, and compare the call with the def line."
    elif etype is ImportError:
        tip = "Your src/ file does not have this name yet (or it is spelled differently) → paste the classes into src/models.py (4.1), Ctrl+S, run again."
    elif etype is AttributeError:
        tip = ("This object has no such attribute or method. Check the spelling and that __init__ saves it as self.<name> = ...\n"
               "   Changed a class? Then create the objects again – old objects keep the old class.")
    else:
        tip = None
    if tip:
        print("\n💡", tip)

try:
    get_ipython().set_custom_exc((ProjectNotReady, NameError, TypeError, AttributeError, ImportError), _friendly_error)
except Exception:
    pass


# ---------------------------------------------------------------- report
print("=" * 70)
print(f"🧪 TEST DATA (Pokémon – NOT your data): {len(TEST)} rows in TEST, {len(DIRTY_RAW)} broken rows in DIRTY_RAW")
print("Your project:", PROJECT)
if PROJECT_OK:
    print(f"  ✅ Lab 2: loader.py reads your data → {len(MY_ROWS)} rows in MY_ROWS")
    print("  ✅ Lab 4: stats.py is there")
    print("  ✅ Lab 5: models.py classes work, main.ipynb uses src.models")
    _text = [c for c, t in MY_TYPES.items() if t in (str, bool)]
    _nums = [c for c, t in MY_TYPES.items() if t in (int, float)]
    print("  Your columns – text:", _text, "| numbers:", _nums)
    print("🎉 Project ready → you can do all parts.")
else:
    print("❌ Your project is NOT ready for Lab 6. Fix these points first (in this order):")
    for i, p in enumerate(PROBLEMS, 1):
        print(f"  {i}. {p}")
    print("➡ Part 2 works anyway (TEST DATA) – you can start it now.")
    print("  Parts 3–4 wait until the list is empty: fix it, then run this Setup cell again.")
print("=" * 70)

🧪 TEST DATA (Pokémon – NOT your data): 800 rows in TEST, 15 broken rows in DIRTY_RAW
Your project: /Users/wintkaykhinemyint/PyDataProject
  ✅ Lab 2: loader.py reads your data → 27901 rows in MY_ROWS
  ✅ Lab 4: stats.py is there
  ✅ Lab 5: models.py classes work, main.ipynb uses src.models
  Your columns – text: ['Gender', 'Sleep Duration', 'Depression'] | numbers: ['Age', 'CGPA', 'Financial Stress']
🎉 Project ready → you can do all parts.


## 2. Lecture 6 step by step (🧪 TEST DATA – not your data)
`DIRTY_RAW` (made by Setup) = 15 Pokémon rows **exactly as they come from a CSV file**: every value is **text**, and some are broken on purpose. Run the cell and find the problems with your eyes first.

In [5]:
for row in DIRTY_RAW:
    print(row)
# problems to find: "fifty-two", "" (empty), "1 00", "maybe", "-80", "NA", "nan", "?", "81,0"

{'Name': 'Bulbasaur', 'Type 1': 'Grass', 'HP': '45', 'Attack': '49', 'Defense': '49', 'Speed': '45', 'Legendary': 'False'}
{'Name': 'Ivysaur', 'Type 1': 'Grass', 'HP': '60', 'Attack': '62', 'Defense': '63', 'Speed': '60', 'Legendary': 'no'}
{'Name': 'Venusaur', 'Type 1': 'Grass', 'HP': '80', 'Attack': '82', 'Defense': '83', 'Speed': '80', 'Legendary': '0'}
{'Name': 'Charmander', 'Type 1': 'Fire', 'HP': '39', 'Attack': 'fifty-two', 'Defense': '43', 'Speed': '65', 'Legendary': 'False'}
{'Name': 'Charmeleon', 'Type 1': 'Fire', 'HP': '', 'Attack': '64', 'Defense': '58', 'Speed': '80', 'Legendary': 'FALSE'}
{'Name': 'Charizard', 'Type 1': 'Fire', 'HP': '78', 'Attack': '84', 'Defense': '78', 'Speed': '1 00', 'Legendary': 'f'}
{'Name': 'Squirtle', 'Type 1': 'Water', 'HP': '44', 'Attack': '48', 'Defense': '65', 'Speed': '43', 'Legendary': 'maybe'}
{'Name': 'Wartortle', 'Type 1': 'Water', 'HP': '59', 'Attack': '63', 'Defense': '-80', 'Speed': '58', 'Legendary': 'False'}
{'Name': 'Blastoise', 'T

### 2.1 try / except – do not crash on bad text
`float("45")` works, but `float("abc")` **crashes** the program with `ValueError`. With `try/except` you say: *"try this; if this error happens, do that instead"*.

**Example (an age typed by a user):**
```python
def read_age(text):
    try:
        return int(text)          # 1. try this
    except ValueError:            # 2. if int() could not do it ...
        return None               # 3. ... give back None instead of crashing

read_age("21")     # 21
read_age("abc")    # None   – no crash
```
Step by step for `read_age("abc")`: Python enters `try` → `int("abc")` raises `ValueError` → Python **jumps** to `except ValueError` → returns `None`. The program goes on.

Catch only the error you expect (`except ValueError`), not everything – otherwise you hide real bugs.

📌 **Under your nose:** [docs.python.org/3/tutorial/errors.html → 8.3 Handling Exceptions](https://docs.python.org/3/tutorial/errors.html#handling-exceptions)

**✏️ Task 2.1** – write your own `to_number(text)`. It must turn every value from the list below into a number, or into `None` when it is not a number:

| text | result | | text | result |
|---|---|---|---|---|
| `"45"` | `45.0` | | `"abc"` | `None` |
| `"1,5"` | `1.5` | | `""`, `"NA"`, `"nan"` | `None` |
| `"1 299,99 zł"` | `1299.99` | | `None` | `None` |
| `"12,5%"` | `12.5` | | `7` (already a number) | `7.0` |

In [6]:
def to_number(text):
    # if text is None → give back None
    if text is None:
        return None
    # if text is already a number (int or float) → give back float(text)    hint: isinstance(text, (int, float))
    if isinstance(text,(int,float)):
        return float(text)
    # clean the text in one line: str(...), strip, remove " ", "%" and "zł", change "," into "."
    text = (
        str(text).strip().replace(" ", "").replace("%","").replace("zł","").replace(",",".")
    )
    # if the lower-case text is one of "", "na", "nan", "none", "-", "?" → give back None
    if text.lower() in  ("","na","nan","none","-","?"):
        return None
    # try: give back float(text) · except ValueError: give back None
    try:
        return float(text)
    except ValueError:
        return None
    # 💡 Stuck? The example read_age above, and to_number in your src/loader.py (Lab 2, PART A) – Lab 4, 2.3

for t in ["45", "1,5", "1 299,99 zł", "12,5%", "abc", "", "NA", "nan", None, 7]:
    print(repr(t), "→", to_number(t))

'45' → 45.0
'1,5' → 1.5
'1 299,99 zł' → 1299.99
'12,5%' → 12.5
'abc' → None
'' → None
'NA' → None
'nan' → None
None → None
7 → 7.0


In [7]:
# @title ✔️ Check 2.1
check("2.1 to_number works for all 11 cases", lambda: not test_to_number(to_number), "look at the cases with ❌ below", safe_test(test_to_number, to_number))
summary("2.1")

✅ 2.1 to_number works for all 11 cases

1/1 OK  🎉


### 2.2 to_bool and converting a whole row
**✏️ Task 2.2 a** – `to_bool(text)`: `"True"`, `"T"`, `"yes"`, `"y"`, `"1"` → `True` · `"False"`, `"F"`, `"no"`, `"n"`, `"0"` → `False` · anything else (`"maybe"`, `""`, `None`) → `None`. Big or small letters do not matter.

**✏️ Task 2.2 b** – `convert_pokemon(raw)`: one text row → one row with real types. `Name` and `Type 1` stay text, but an empty text `""` becomes `None`. `HP`, `Attack`, `Defense`, `Speed` go through **your** `to_number`, `Legendary` through **your** `to_bool`. Then convert all 15 rows into `dirty_rows`.

In [8]:
def to_bool(text):
    # make value = the text as str, without spaces at the ends, in small letters
    value = str(text).strip().lower()
    # if value is one of "true", "t", "yes", "y", "1" → give back True
    if value in ("true", "t", "yes", "y", "1"):
        return True
    # if value is one of "false", "f", "no", "n", "0" → give back False
    if value in ("false", "f", "no", "n", "0"):
        return False 
    # otherwise give back None
    return None


def convert_pokemon(raw):
    # give back a NEW dictionary with the same 7 keys:
    # "Name": raw["Name"] or None   ← "or None" turns "" into None
    # "Type 1": the same way · "HP", "Attack", "Defense", "Speed": to_number(raw[...]) · "Legendary": to_bool(raw[...])
    return {
        "Name" : raw["Name"] or None,
        "Type 1" : raw["Type 1"] or None,
        "HP" : to_number(raw["HP"]),
        "Attack": to_number(raw["Attack"]),
        "Defense": to_number(raw["Defense"]),
        "Speed": to_number(raw["Speed"]),
        "Legendary": to_bool(raw["Legendary"]),
    }


# dirty_rows = convert_pokemon(...) of every row of DIRTY_RAW (a loop with append, or one line)
dirty_rows = [convert_pokemon(row) for row in DIRTY_RAW]
for row in dirty_rows:
    print(row)
# 💡 Stuck? to_bool in your src/loader.py (Lab 2) · a list from a loop: Lab 5, 2.1 b step 6 / 2.3 b

{'Name': 'Bulbasaur', 'Type 1': 'Grass', 'HP': 45.0, 'Attack': 49.0, 'Defense': 49.0, 'Speed': 45.0, 'Legendary': False}
{'Name': 'Ivysaur', 'Type 1': 'Grass', 'HP': 60.0, 'Attack': 62.0, 'Defense': 63.0, 'Speed': 60.0, 'Legendary': False}
{'Name': 'Venusaur', 'Type 1': 'Grass', 'HP': 80.0, 'Attack': 82.0, 'Defense': 83.0, 'Speed': 80.0, 'Legendary': False}
{'Name': 'Charmander', 'Type 1': 'Fire', 'HP': 39.0, 'Attack': None, 'Defense': 43.0, 'Speed': 65.0, 'Legendary': False}
{'Name': 'Charmeleon', 'Type 1': 'Fire', 'HP': None, 'Attack': 64.0, 'Defense': 58.0, 'Speed': 80.0, 'Legendary': False}
{'Name': 'Charizard', 'Type 1': 'Fire', 'HP': 78.0, 'Attack': 84.0, 'Defense': 78.0, 'Speed': 100.0, 'Legendary': False}
{'Name': 'Squirtle', 'Type 1': 'Water', 'HP': 44.0, 'Attack': 48.0, 'Defense': 65.0, 'Speed': 43.0, 'Legendary': None}
{'Name': 'Wartortle', 'Type 1': 'Water', 'HP': 59.0, 'Attack': 63.0, 'Defense': -80.0, 'Speed': 58.0, 'Legendary': False}
{'Name': 'Blastoise', 'Type 1': 'Wat

In [9]:
# @title ✔️ Check 2.2
check("2.2 a to_bool works for all cases", lambda: not test_to_bool(to_bool), "look at the cases below", safe_test(test_to_bool, to_bool))
check("2.2 b convert_pokemon + dirty_rows", lambda: dirty_rows == REF_ROWS,
      "15 rows; Name '' → None; numbers through to_number; Legendary through to_bool")
summary("2.2")

✅ 2.2 a to_bool works for all cases
✅ 2.2 b convert_pokemon + dirty_rows

2/2 OK  🎉


### 2.3 raise – YOUR code reports a problem
`try/except` **catches** errors. `raise` **creates** one: *"this row is wrong, and here is why"*.

**Example (a shop order):**
```python
def check_order(order):
    if order["amount"] <= 0:
        raise ValueError("amount must be positive")      # stop here and report

try:
    check_order({"amount": -2})
except ValueError as e:              # e = the error; str(e) = its message
    print("rejected:", e)            # rejected: amount must be positive
```
`except ValueError as e` gives you the error as a variable – `str(e)` is the message you wrote in `raise`.

📌 **Under your nose:** [errors.html → 8.4 Raising Exceptions](https://docs.python.org/3/tutorial/errors.html#raising-exceptions)

**✏️ Task 2.3 a** – `check_row(row, required, non_negative)`:
- `required` = a list of columns that must not be `None` → otherwise `raise ValueError(f"{column} is missing")`
- `non_negative` = a list of number columns that must not be below 0 → otherwise `raise ValueError(f"{column} is negative")`
- a good row: the function just ends (gives back nothing).

The column names are **parameters** – so the same function will work for your data in part 3.

**✏️ Task 2.3 b** – a loop over `dirty_rows` that counts in `report` how many rows were rejected for each reason.

**Counting in a dictionary – how `report.get(message, 0) + 1` works:**
`report` starts empty: `{}`. For every bad row you get a `message` (the text from `raise`, e.g. `"HP is missing"`) and you want to add 1 to its counter.
- `report.get(message, 0)` = *"give me the current count for this message – and if this message is not in the dictionary yet, give me 0"*.
- `+ 1` = one more.
- `report[message] = ...` = save the new count back.

| bad row | message | `report.get(message, 0)` | saved | report after |
|---|---|---|---|---|
| Charmander | `"Attack is missing"` | 0 (not there yet) | 1 | `{"Attack is missing": 1}` |
| Charmeleon | `"HP is missing"` | 0 | 1 | `{..., "HP is missing": 1}` |
| *(another HP problem)* | `"HP is missing"` | 1 | 2 | `{..., "HP is missing": 2}` |

Why not just `report[message] += 1`? The first time a message appears it is not in the dictionary yet → `KeyError`. `.get(..., 0)` avoids that. (It is the same trick as in `count_by` in your `stats.py`.)

In [13]:
def check_row(row, required, non_negative):
    # for every column in required:
    #     if row[column] is None → raise ValueError(f"{column} is missing")
    for column in required:
        if row[column] is None:
            raise ValueError(f"{column} is missing")
    # for every column in non_negative:
    #     if the value is not None AND it is below 0 → raise ValueError(f"{column} is negative")
    for column in non_negative:
        value = row[column]
        if value is not None and value < 0:
            raise ValueError(f"{column} is negative")


REQUIRED = ["Name", "Type 1", "HP", "Attack"]      # rules for the test data
NON_NEGATIVE = ["Defense", "Speed"]

report = {}
good = 0
for row in dirty_rows:
    # try:  call check_row(row, REQUIRED, NON_NEGATIVE)  and then add 1 to good
    # except ValueError as e:
    #     message = the text of the error:  str(e)
    #     add 1 to the counter of this message:  report[message] = report.get(message, 0) + 1   (see the table above)
    try:
        check_row(row, REQUIRED, NON_NEGATIVE)
        good += 1
    except ValueError as error:
        message = str(error)
        report[message] = report.get(message,0) + 1
print("good rows:", good)        # expected: 10
print(report)
# 💡 Stuck? The example check_order above · counting in a dictionary: count_by in your src/stats.py (Lab 4)

good rows: 10
{'Attack is missing': 1, 'HP is missing': 1, 'Defense is negative': 1, 'Name is missing': 1, 'Type 1 is missing': 1}


In [14]:
# @title ✔️ Check 2.3
check("2.3 a check_row raises the right errors", lambda: not test_check_row(check_row), "see the points below", safe_test(test_check_row, check_row))
check("2.3 b the loop: 10 good rows and the report", lambda: good == 10 and report == REPORT_TEST, f"report should be {REPORT_TEST}")
summary("2.3")

✅ 2.3 a check_row raises the right errors
✅ 2.3 b the loop: 10 good rows and the report

2/2 OK  🎉


### 2.4 Debugging – find the bug, do not guess
The function below should give the mean Attack of one type. It has **3 bugs**. Run the cell – it shows the error message (the `try/except` around it only keeps the notebook running).

**How to read an error message (bottom → top):**
1. **Last line** = what happened: e.g. `TypeError: unsupported operand type(s) for +=: 'int' and 'NoneType'` → you add `None` to a number.
2. **The line with `---->`** (or `line 6`) = **where** it happened.

**Three ways to look inside:**
- **print:** add `print(row["Name"], total)` inside the loop and run again.
- **the debugger after a crash:** in a **new cell** type `%debug` and run it. A small box appears – type commands there:
  `p row["Name"]` (print a value) · `p total` · `l` (show the code around) · `q` (quit). This works in Colab and in Jupyter.
- **breakpoint():** put the line `breakpoint()` in the code – Python stops there and opens the same box (`n` = next line, `c` = continue).

📌 **Under your nose:** [docs.python.org/3/library/pdb.html → Debugger Commands](https://docs.python.org/3/library/pdb.html#debugger-commands)

In [ ]:
def mean_by_type(rows, type1):          # ⚠️ this function has 3 bugs – do not change it, find them
    total = 0
    count = 0
    for row in rows:
        if row["Type 1"] == type1:
            total += row["Attack"]
        count += 1
    return round(total / count, 2)

try:
    print(mean_by_type(dirty_rows, "Fire"))
except Exception as e:
    LAST_ERROR = e
    traceback.print_exc()                  # shows the error message like a normal crash

# Debugger: put the crash into the debugger – remove the # in the next line, run, type  p row["Name"]  then  p total  then  q
# import pdb; pdb.post_mortem(LAST_ERROR.__traceback__)

**✏️ Task 2.4 a** – write what you found (with `print` or with the debugger – your choice):
- `crash_name` – the `"Name"` of the row at which the function crashed,
- `crash_total` – the value of `total` at that moment.

**✏️ Task 2.4 b** – write `mean_by_type_fixed` without the 3 bugs:
1. a row whose `"Attack"` is `None` is **skipped**,
2. `count` counts **only the rows of that type** (look at the indentation of `count += 1`!),
3. if there is not a single row of that type → give back `None` instead of dividing by 0.

Expected: `"Fire"` → `74.0`, `"Water"` → `79.75`, `"Dragon"` → `None`.

In [ ]:
# the Name of the row where it crashed (text in "...")
crash_name = "Charmander"
# the value of total at that moment (a number)
crash_total = 0


def mean_by_type_fixed(rows, type1):
    total = 0
    count = 0
    for row in rows:
        # if the row is of type1 AND its "Attack" is not None:
        #     add the Attack to total and add 1 to count (BOTH inside this if)
        if row in rows:
            if row["Type 1"] == type1 and row["Attack"] is not None:
                total += row["Attack"]
                count += 1
    # if count is 0 → give back None
    if count == 0:
        return None
    return round(total / count, 2)

print(mean_by_type_fixed(dirty_rows, "Fire"), mean_by_type_fixed(dirty_rows, "Water"), mean_by_type_fixed(dirty_rows, "Dragon"))
# expected: 74.0 79.75 None
# 💡 Stuck? mean() in your src/stats.py (Lab 4) skips None and checks for an empty list – the same idea

In [ ]:
# @title ✔️ Check 2.4
check("2.4 a crash_name and crash_total", lambda: (crash_name, crash_total) == ("Charmander", 0),
      'add print(row["Name"], total) in the loop, or use the debugger: p row["Name"], p total')
check("2.4 b mean_by_type_fixed", lambda: (mean_by_type_fixed(REF_ROWS, "Fire"), mean_by_type_fixed(REF_ROWS, "Water"), mean_by_type_fixed(REF_ROWS, "Dragon")) == (74.0, 79.75, None),
      "skip None, count only rows of the type (indentation!), return None when count == 0")
summary("2.4")

### 2.5 Iterators and generators
**Iterator** = something that gives you **the next element** each time you ask. A `for` loop does exactly this for you:
```python
it = iter(["a", "b"])     # make an iterator from a list
next(it)                  # 'a'
next(it)                  # 'b'
next(it)                  # StopIteration – nothing left (for knows to stop here)
```
**Generator** = a function with `yield` instead of `return`. It does **not** build a whole list – it gives one element, **pauses**, and continues when asked for the next one.

**Example:**
```python
def even_numbers(numbers):
    for n in numbers:
        if n % 2 == 0:
            yield n            # give n, pause here

gen = even_numbers([1, 2, 3, 4])
next(gen)                      # 2      (runs until the first yield)
next(gen)                      # 4      (continues after the pause)
list(even_numbers([1, 2, 3, 4]))   # [2, 4]   – list() asks for all of them
```
Why it matters for data: 1 000 000 rows → a generator holds only **one row at a time** in memory.

📌 **Under your nose:** [tutorial/classes.html → 9.8 Iterators](https://docs.python.org/3/tutorial/classes.html#iterators) and [9.9 Generators](https://docs.python.org/3/tutorial/classes.html#generators)

**🤔 Predict** the three results in the comments, then run.

In [ ]:
def names_with_letter(rows, letter):
    for row in rows:
        if row["Name"] and letter in row["Name"]:
            yield row["Name"]

gen = names_with_letter(REF_ROWS, "saur")
print(next(gen))                                        # my guess: ...
print(next(gen))                                        # my guess: ...
print(list(names_with_letter(REF_ROWS, "x")))           # my guess: ...

**✏️ Task 2.5** – `valid_rows(rows, report, required, non_negative)` – a **generator** that gives only the good rows and counts the bad ones in `report` (like your loop in 2.3 b, but with `yield`).

Expected on `dirty_rows`: 10 rows come out, `report` = the same as in 2.3 b.

In [ ]:
def valid_rows(rows, report, required, non_negative):
    for row in rows:
        # try: call check_row(row, required, non_negative)
        # except ValueError as e:  message = str(e),  count it like in 2.3 b,  then  continue  (skip to the next row)
        ...
        # the row is good → give it out with yield
        ...


test_report = {}
first = next(valid_rows(dirty_rows, {}, REQUIRED, NON_NEGATIVE))      # only the FIRST good row
print(first["Name"])                                                   # expected: Bulbasaur
kept = list(valid_rows(dirty_rows, test_report, REQUIRED, NON_NEGATIVE))
print(len(kept), test_report)                                          # expected: 10 and the report from 2.3 b
# 💡 Stuck? It is your loop from 2.3 b – only "good += 1" becomes "yield row", and after the except comes "continue"

In [ ]:
# @title ✔️ Check 2.5
check("2.5 valid_rows is a generator that keeps 10 rows", lambda: not test_valid_rows(valid_rows), "see the points below", safe_test(test_valid_rows, valid_rows))
summary("2.5")

### 2.6 Decorators – add something to a function without changing it
A **decorator** is a function that takes a function and gives back a **new, wrapped** version of it. `@name` above `def` puts the wrapper on.

**Example:**
```python
def shout(func):                     # 1. takes a function
    def wrapper(*args, **kwargs):    # 2. a new function that accepts ANY arguments
        result = func(*args, **kwargs)   # 3. call the original
        return result.upper()            # 4. add something: make it LOUD
    return wrapper                   # 5. give back the new function

@shout                               # the same as:  greet = shout(greet)
def greet(name):
    return f"hello {name}"

greet("Ola")                         # 'HELLO OLA'
```
`*args, **kwargs` = "whatever arguments come, pass them on". `func.__name__` = the name of the function as text (`"greet"`).

Typical uses: measuring time, logging, checking access – the same addition for many functions, written once.

📌 **Under your nose:** [glossary → decorator](https://docs.python.org/3/glossary.html#term-decorator) · [time.perf_counter()](https://docs.python.org/3/library/time.html#time.perf_counter)

**✏️ Task 2.6 a** – `timed(func)`: the wrapper measures how long the function runs and prints e.g. `clean took 0.0002 s`, then gives back the result.

**✏️ Task 2.6 b** – `clean(rows, required, non_negative)` decorated with `@timed`: makes an empty report, runs `valid_rows`, and gives back **two things**: the list of good rows and the report.

In [ ]:
import time

def timed(func):
    def wrapper(*args, **kwargs):
        # remember the start time: start = time.perf_counter()
        ...
        # call the original function with *args, **kwargs and keep its result
        result = ...
        # print the function name and the time:  f"{func.__name__} took {time.perf_counter() - start:.4f} s"
        ...
        # give back the result
        return ...
    return wrapper


# put the decorator above the def:  @timed
# ...
def clean(rows, required, non_negative):
    # make an empty dictionary report
    report = ...
    # kept = a LIST of everything valid_rows(...) gives   hint: list(valid_rows(rows, report, required, non_negative))
    kept = ...
    # give back both:  return kept, report
    return ...


kept, report = clean(dirty_rows, REQUIRED, NON_NEGATIVE)     # expected: "clean took 0.0000 s"
print(len(kept), report)
# 💡 Stuck? The example shout above: the same 5 steps, only step 4 measures time instead of .upper()

In [ ]:
# @title ✔️ Check 2.6
check("2.6 a timed prints the name + time and returns the result", lambda: not test_timed(timed), "see the points below", safe_test(test_timed, timed))
check("2.6 b clean is @timed and returns (kept, report)", lambda: not test_clean(clean), "see the points below", safe_test(test_clean, clean))
summary("2.6")

## 3. Your project
From here on you work with **your** files. In this part you **copy your own code** from part 2 into your files – that is fine, it is your code.

### 3.1 Your own `to_number` and `to_bool` in `src/loader.py`
In Lab 2 these two functions were written for you. Now you replace them with yours.

1. Open `src/loader.py`. **Colab:** 📁 on the left → `drive` → `MyDrive` → `PyDS_project` → `src` → double-click `loader.py`. **Jupyter:** click `src` → `loader.py`.
2. Find `def to_number(text):` (in PART A). Select the **whole** function – from `def` down to its last line `return None` – and delete it.
3. In this notebook copy your `to_number` from 2.1 (from `def` to the end of the function, **without** the `for t in [...]` test lines) and paste it in the same place.
4. Do the same with `to_bool` (your version is in 2.2).
5. Save: **Ctrl+S**. Run Check 3.1.

💡 Stuck? Delete exactly one function – the next `def` (e.g. `def _convert`) must stay. The indentation of `def` = 0 spaces, like the other functions.

In [ ]:
# @title ✔️ Check 3.1 – my src/loader.py
require_project()
try:
    _L = fresh("loader")
    _err = None
except Exception as e:
    _L, _err = None, e
if _err is not None:
    print(f"❌ src/loader.py cannot be imported: {type(_err).__name__}: {_err}")
    if getattr(_err, "lineno", None):
        print(f"   → look at line {_err.lineno} of src/loader.py (and the line above it) – often a wrong indentation after pasting.")
    RESULTS["3.1 loader.py imports"] = False
else:
    check("3.1 loader.py: to_number is YOUR version", lambda: not test_to_number(_L.to_number),
          "paste your to_number from 2.1 in place of the old one (the old one gives None for '1 299,99 zł')", safe_test(test_to_number, _L.to_number))
    check("3.1 loader.py: to_bool works", lambda: not test_to_bool(_L.to_bool), "paste your to_bool from 2.2", safe_test(test_to_bool, _L.to_bool))
    check("3.1 your data still loads", lambda: len(_L.load_project_data()) == len(MY_ROWS), "load_project_data() must still work – check what you deleted")
    MY_ROWS = _L.load_project_data()
    print("\nEmpty values (None) in YOUR data after loading – useful for 3.2:")
    for _c in MY_TYPES:
        print(f"   {_c}: {sum(1 for r in MY_ROWS if r[_c] is None)}")
summary("3.1")

### 3.2 `src/cleaning.py` – your cleaning module
1. Open `src/cleaning.py` (the same way as `loader.py`). Write everything **below** the line `# Your code from Lab 5 goes below.` (the number is from the old plan – it does not matter).
2. First line: `import time`
3. Copy from this notebook **your** functions (only the `def` blocks, no test lines): `check_row` (2.3) → `valid_rows` (2.5) → `timed` and `clean` (2.6, together with `@timed`).
4. Below them write **two lists for YOUR data** (the column names exactly as Setup printed them):
   - `REQUIRED` – the columns that **must** have a value, e.g. the columns you group by and the numbers you analyse,
   - `NON_NEGATIVE` – the **number** columns that cannot be below 0, e.g. distance, price, age, count.
5. Save: **Ctrl+S**. Run Check 3.2.

The Check above (3.1) printed how many empty values each of your columns has – use it to choose `REQUIRED`.

💡 Stuck? `REQUIRED` and `NON_NEGATIVE` look exactly like the two lists in 2.3, only with your column names.

In [ ]:
# @title ✔️ Check 3.2 – my src/cleaning.py
require_project()
try:
    CLEANING = fresh("cleaning")
    _err = None
except Exception as e:
    CLEANING, _err = None, e
if _err is not None:
    print(f"❌ src/cleaning.py cannot be imported: {type(_err).__name__}: {_err}")
    if getattr(_err, "lineno", None):
        print(f"   → look at line {_err.lineno} of src/cleaning.py (and the line above it).")
    if isinstance(_err, NameError):
        print("   → is 'import time' at the top? Are the functions pasted in the order check_row, valid_rows, timed, clean?")
    RESULTS["3.2 cleaning.py imports"] = False
else:
    for _label, _fn, _test in [("3.2 check_row", "check_row", test_check_row), ("3.2 valid_rows", "valid_rows", test_valid_rows),
                               ("3.2 timed", "timed", test_timed), ("3.2 clean", "clean", test_clean)]:
        _bad = safe_test(_test, getattr(CLEANING, _fn)) if hasattr(CLEANING, _fn) else [f"there is no function {_fn} in src/cleaning.py"]
        check(_label + " in cleaning.py", lambda: not _bad, "fix it in src/cleaning.py, Ctrl+S, run this check again", _bad)
    _req, _nn = getattr(CLEANING, "REQUIRED", None), getattr(CLEANING, "NON_NEGATIVE", None)
    check("3.2 REQUIRED = a list of YOUR columns", lambda: isinstance(_req, list) and _req and all(c in MY_TYPES for c in _req),
          f"REQUIRED = [...] with names from your columns: {list(MY_TYPES)}")
    check("3.2 NON_NEGATIVE = YOUR number columns", lambda: isinstance(_nn, list) and _nn and all(MY_TYPES.get(c) in (int, float) for c in _nn),
          f"only number columns (int/float): {[c for c, t in MY_TYPES.items() if t in (int, float)]}")
summary("3.2")

### 3.3 Try it on your data (here)
Run **your** `clean` from **your** `src/cleaning.py` on **your** rows `MY_ROWS` (Setup loaded them).

In [ ]:
require_project()
fresh("cleaning")                                       # the newest version of YOUR src/cleaning.py
from src.cleaning import clean, REQUIRED, NON_NEGATIVE

# run clean on MY_ROWS with REQUIRED and NON_NEGATIVE; keep the two results as my_clean and my_report
my_clean, my_report = ...
# print: how many rows are left, out of how many (len of both lists)
...
# print the report
...
# 💡 Stuck? It is the last two lines of 2.6 – only MY_ROWS and your lists instead of dirty_rows

In [ ]:
# @title ✔️ Check 3.3
require_project()
check("3.3 my_clean + removed rows = all rows", lambda: len(my_clean) + sum(my_report.values()) == len(MY_ROWS),
      "my_clean, my_report = clean(MY_ROWS, REQUIRED, NON_NEGATIVE)")
summary("3.3")

### 3.4 Write it into `main.ipynb`
Open `main.ipynb` (Colab: **File → Open notebook → Google Drive**; Jupyter: click it). Find the section **`## Lab 5 – Errors and cleaning (src/cleaning.py)`** and its cell `# Lab 5 – your code here`. Write there, line by line:

1. import `clean`, `REQUIRED`, `NON_NEGATIVE` from `src.cleaning`
2. run `clean` on **`rows`** (the variable from the Lab 2 section) with your two lists; save the results as `clean_rows` and `report`
3. print: how many rows are left of how many
4. print `report`
5. make a `Dataset` from `clean_rows` (your class from Lab 5 – it is already imported in the section above) and print the report of **one** of your analyses on it
6. a comment `# Finding: ...` – what was removed, and did it change your result?

Then **Run all** (Colab: Runtime → Run all; Jupyter: Kernel → Restart & Run All) and come back to run Check 3.4.

💡 Stuck? Lines 1–4 = your cell 3.3 (only `rows` instead of `MY_ROWS`). Line 5 = your Lab 5 section in `main.ipynb`, just one section higher.

In [ ]:
# @title ✔️ Check 3.4 – my main.ipynb
require_project()
_cells = ["".join(c.get("source", [])) for c in json.loads((PROJECT / "main.ipynb").read_text(encoding="utf-8"))["cells"]
          if c.get("cell_type") == "code" and "src.cleaning" in "".join(c.get("source", []))]
_code = main_code()
check("3.4 main.ipynb imports from src.cleaning", lambda: "src.cleaning" in _code, "line 1: from src.cleaning import clean, REQUIRED, NON_NEGATIVE")
MAIN_OK, _msg, MAIN_NS = run_main()
check("3.4 main.ipynb runs from top to bottom", lambda: MAIN_OK, f"error in {_msg} → fix it in main.ipynb and Run all there")
check("3.4 main.ipynb has clean rows and a report", lambda: any(isinstance(v, dict) for k, v in MAIN_NS.items() if "report" in k.lower()) and isinstance(MAIN_NS.get("clean_rows"), list),
      "clean_rows, report = clean(rows, REQUIRED, NON_NEGATIVE)")
check("3.4 # Finding: in the cleaning section", lambda: any("finding:" in c.lower() for c in _cells), "add a line  # Finding: ...  in the same cell")
summary("3.4")

### 3.5 Finding and three explanations (your own words, full sentences)

In [ ]:
FINDING = "..."
# what did cleaning show about YOUR data (the same as # Finding in main)?

EXPLAIN_TRY = "..."
# what does try/except do in YOUR to_number?

EXPLAIN_GENERATOR = "..."
# how is valid_rows (yield) different from a normal function with return?

EXPLAIN_DECORATOR = "..."
# what happens when you call clean(...) that has @timed above it?

## 4. Result check
Run at the end. It checks 6 points and saves `docs/lab06_check.md` (the only file this notebook writes). Then push to GitHub, if your team uses it.

🏆 **For those who finish today:** show the teacher **✅ 6/6**.

In [ ]:
# @title ✔️ 4. Result check – run at the end
require_project()
def _part_ok(prefixes):
    missing = [p for p in prefixes if not any(l.startswith(p) for l in RESULTS)]
    bad = [l for l in RESULTS if any(l.startswith(p) for p in prefixes) and not RESULTS[l]]
    return missing, bad

def _words(text):
    if not isinstance(text, str) or text.strip() in ("", "..."):
        return 0
    return len(text.split())

items = []
_m, _b = _part_ok(["2.1", "2.2", "2.3", "2.4", "2.5", "2.6"])
items.append(("Part 2 – try/except, raise, debugging, generator, decorator (TEST DATA)", not _m and not _b,
              f"run the check cells of: {_m}" if _m else f"fix: {_b}"))
try:
    _L = fresh("loader")
    _bad = test_to_number(_L.to_number) + test_to_bool(_L.to_bool)
    _rows = _L.load_project_data()
except Exception as e:
    _bad, _rows = [f"{type(e).__name__}: {e}"], []
items.append(("src/loader.py – your own to_number and to_bool, your data loads", not _bad and len(_rows) > 0, "; ".join(_bad[:2]) + " (see 3.1)"))
try:
    _C = fresh("cleaning")
    _bad = test_check_row(_C.check_row) + test_valid_rows(_C.valid_rows) + test_timed(_C.timed) + test_clean(_C.clean)
    if not (_C.REQUIRED and all(c in MY_TYPES for c in _C.REQUIRED) and all(MY_TYPES.get(c) in (int, float) for c in _C.NON_NEGATIVE)):
        _bad.append("REQUIRED / NON_NEGATIVE must be lists of your columns")
except Exception as e:
    _bad = [f"{type(e).__name__}: {e}"]
items.append(("src/cleaning.py – check_row, valid_rows, @timed clean + rules for your columns", not _bad, "; ".join(_bad[:2]) + " (see 3.2)"))
_code = main_code()
_ok, _msg, _ns = run_main()
items.append(("main.ipynb uses src.cleaning (clean_rows, report)", "src.cleaning" in _code and isinstance(_ns.get("clean_rows"), list), "section 'Lab 5 – Errors and cleaning' (see 3.4)"))
items.append(("main.ipynb runs from top to bottom", _ok, _msg))
_texts = {k: globals().get(k) for k in ["FINDING", "EXPLAIN_TRY", "EXPLAIN_GENERATOR", "EXPLAIN_DECORATOR"]}
items.append(("Finding + three explanations (at least 8 words each)", all(_words(t) >= 8 for t in _texts.values()), "fill the four texts in 3.5"))

score = sum(ok for _, ok, _ in items)
print("=" * 70)
for name, ok, hint in items:
    print(("✅ " if ok else "❌ ") + name + ("" if ok else "\n     → " + hint))
print("=" * 70)
print(f"RESULT: {score}/{len(items)}" + ("  🎉 Lab 6 done!" if score == len(items) else "  → fix the ❌ and run this cell again"))

report_md = [f"# Lab 6 – result check ({datetime.date.today()})", "", f"**Result: {score}/{len(items)}**", ""]
report_md += [f"- {'[OK]' if ok else '[X]'} {name}" for name, ok, _ in items]
for k, v in _texts.items():
    report_md += ["", f"## {k}", str(v)]
try:
    (PROJECT / "docs").mkdir(exist_ok=True)
    (PROJECT / "docs" / "lab06_check.md").write_text("\n".join(report_md) + "\n", encoding="utf-8")
    print("📝 saved: docs/lab06_check.md")
except Exception as e:
    print("Could not save the report:", e)

## What you have after Lab 6
- `src/loader.py` – `to_number` and `to_bool` are now **yours** (with try/except).
- `src/cleaning.py` – `check_row` (raise), `valid_rows` (generator), `@timed` `clean` (decorator) and the rules for your data.
- `main.ipynb` – a cleaning section: what was removed and why. Project chain: `loader → cleaning → stats → models → main`.

### Sources
- Errors and exceptions (try/except, raise): https://docs.python.org/3/tutorial/errors.html
- The Python debugger pdb: https://docs.python.org/3/library/pdb.html
- Iterators and generators: https://docs.python.org/3/tutorial/classes.html#iterators
- Decorator (glossary): https://docs.python.org/3/glossary.html#term-decorator
- time.perf_counter: https://docs.python.org/3/library/time.html#time.perf_counter
- 🧪 TEST DATA – Pokémon dataset from the repository KeithGalli/pandas: https://github.com/KeithGalli/pandas